<div align="center">

# Modelo de Churn / Attrition Bancario

**Panel temporal CIF × mes → feature engineering → modelos → ensemble → explicabilidad**

</div>

---

Este notebook es **autocontenido**: lleva dentro todo el pipeline, del contrato de
datos a la lista de retención. No clona ningún repositorio ni importa nada externo.
La metodología —por qué el target es el que es, por qué el split lleva embargo, qué
se pide al agente SQL— está en el `README.md`.

## Cómo usarlo

Ejecuta **Parte I** entera (define el pipeline, no calcula nada) y después **Parte II**
en orden. Lo único que hay que tocar está en la celda de configuración.

| `data.source` | Qué hace | Cuándo |
|:---|:---|:---|
| `synthetic` | Genera datos simulados en memoria | **Empieza aquí.** Valida el pipeline sin datos reales |
| `drive` | Lee los extractos del agente SQL desde Drive | Cuando tengas las tablas |

## Índice

**Parte I — El pipeline** (definiciones)

| | |
|:--|:--|
| 1 | Configuración |
| 2 | Contrato de datos |
| 3 | Generador sintético |
| 4 | Carga, despivote y validación |
| 5 | Panel, ownership y target |
| 6 | Feature engineering |
| 7 | Split, métricas y calibración |
| 8 | Modelos |
| 9 | Explicabilidad |

**Parte II — El análisis** (ejecución)

| | |
|:--|:--|
| 10 | Carga y validación |
| 11 | EDA |
| 12 | Panel y target |
| 13 | Features |
| 14 | Split out-of-time |
| 15 | Tuning |
| 16 | Stacking |
| 17 | Red secuencial (GRU) |
| 18 | Resultados |
| 19 | Calibración |
| 20 | Explicabilidad (SHAP) |
| 21 | Contrafactuales |
| 22 | Lista de scoring |

## Dónde van los datos reales

En `MyDrive/churn_model/data/raw/`, con estos nombres (configurables abajo):

| Query | Archivo | Contenido |
|:---|:---|:---|
| Q1 | `q1_party_master.csv` | Maestro de clientes |
| Q2 | `q2_cif_account_relationship.csv` | Relación CIF ↔ cuenta y rol |
| Q3 | `q3_deposit_monthly.csv` | Saldo de depósitos a fin de mes |
| Q4 | `q4_loan_monthly.csv` | Préstamos vigentes por mes |
| Q5 | `q5_txn_monthly.csv` | Actividad transaccional |
| Q6 | `q6_digital_contact_monthly.csv` | Canales digitales y contactos |
| Q7 | `q7_product_events.csv` | Aperturas, cierres y eventos |

> **Q4–Q7 son opcionales.** Puedes empezar con Q1–Q3 y el pipeline lo tolera.

> **Si bajas por ODBC a Excel**, pide las tablas mensuales en formato ancho —un mes
> por columna, `eom_balance_2024_07`— y el maestro de clientes con **una fila por
> CIF**. El pipeline detecta la orientación solo. El detalle está en el README.

# Parte I — El pipeline

Todo lo que sigue son **definiciones**: ejecutarlo no calcula nada. El análisis
empieza en la Parte II.

## 0 · Setup

In [ ]:
# Colab ya trae numpy/pandas/sklearn/torch/xgboost
!pip install -q lightgbm catboost shap 2>/dev/null
print("✓ dependencias listas")

In [ ]:
# ── Montar Google Drive ──
# Fuera de Colab esto no hace nada, así que la misma celda sirve en local.
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✓ Drive montado")
else:
    print("ℹ️  Fuera de Colab — se omite el montaje")

In [ ]:
import re
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform, randint, uniform
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             log_loss, roc_auc_score)
from sklearn.model_selection import (GroupShuffleSplit, RandomizedSearchCV,
                                     StratifiedGroupKFold)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, OneHotEncoder,
                                   OrdinalEncoder, StandardScaler)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 10})
print("✓ entorno listo")

## 1 · Configuración

**Esta es la única celda que hay que tocar.** Todo el comportamiento del pipeline
sale de aquí: de dónde vienen los datos, cómo se define el target, cuánta historia
mira cada feature, qué modelos entran y cómo se explican.

Los dos parámetros que más mueven el resultado, y que hay que fijar **con los datos
delante** (la EDA de la Parte II está para eso):

- `target.low_balance_abs` — el umbral de saldo bajo. El valor de fábrica ($100)
  viene de los datos sintéticos y casi seguro no es el tuyo.
- `panel.performance_window_months` — a cuántos meses vista se predice. Más largo
  captura más fugas pero deja menos margen de reacción.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — la única celda que hay que tocar
# ════════════════════════════════════════════════════════════════════════
CONFIG = {

  "data": {
    # synthetic = genera datos simulados en memoria (para probar el pipeline)
    # drive     = lee los extractos del agente SQL desde Google Drive
    "source": "synthetic",                      # ⬅️ CAMBIA ESTO con datos reales

    "drive_dir":     "/content/drive/MyDrive/churn_model/data/raw",
    "processed_dir": "/content/drive/MyDrive/churn_model/data/processed",
    "artifacts_dir": "/content/drive/MyDrive/churn_model/artifacts",
    "file_format": "csv",                       # csv | parquet

    # Nombre de archivo (sin extensión) de cada una de las 7 queries.
    "files": {
        "party":         "q1_party_master",
        "relationships": "q2_cif_account_relationship",
        "deposits":      "q3_deposit_monthly",
        "loans":         "q4_loan_monthly",
        "transactions":  "q5_txn_monthly",
        "digital":       "q6_digital_contact_monthly",
        "events":        "q7_product_events",
    },
    # Tablas que el pipeline tolera si no existen todavía: permite empezar a
    # modelar con Q1-Q3 mientras llegan las demás.
    "optional": ["loans", "transactions", "digital", "events"],
  },

  # ── Estructura del panel temporal ───────────────────────────────────
  # Aritmética a tener presente: el panel utilizable pierde
  #   min_history_months (por detrás, para calcular features)
  # + gap_months + performance_window_months (por delante, para etiquetar)
  # y el embargo del split OOT cuesta otros (gap + performance) meses.
  # Con 36 meses de historia quedan ~20 meses de train.
  "panel": {
    "history_months": 36,             # cuántos meses de snapshots generar
    "feature_window_months": 12,      # ventana de historia para features
    "min_history_months": 6,          # historia mínima exigida
    "gap_months": 1,                  # latencia: hace el modelo accionable
    "performance_window_months": 6,   # horizonte de predicción
    "min_tenure_months": 6,           # excluir clientes demasiado nuevos
  },

  # ── Definición del target ───────────────────────────────────────────
  "target": {
    "low_balance_abs": 100.0,             # ⬅️ AJUSTA con la EDA de saldos
    "low_balance_pct_of_baseline": 0.05,  # o 5% de su propio promedio 12m
    "inactivity_days": 90,                # sin transacciones del cliente
    "persistence_months": 2,              # meses consecutivos para confirmar
    "exclude_involuntary": True,
    "involuntary_reason_codes": [
        "DECEASED", "AML_EXIT", "FRAUD",
        "CHARGE_OFF", "BANK_INITIATED", "ESCHEAT",
    ],
  },

  # ── Pesos de ownership por rol del CIF ──────────────────────────────
  # None = repartir 1/n_owners entre los titulares. Es lo que evita el
  # doble conteo de saldos en cuentas joint.
  "ownership_weights": {
    "primary": 1.0, "sole": 1.0, "joint": None,
    "custodian": 1.0, "trustee": 1.0,
    "authorized_signer": 0.0, "beneficiary": 0.0,
    "borrower": 1.0, "co_borrower": None, "guarantor": 0.0,
  },
  # Roles que hacen a un CIF elegible para la población de modelado
  "eligible_roles": ["primary", "sole", "joint", "custodian", "trustee",
                     "borrower", "co_borrower"],

  # ── Split y validación ──────────────────────────────────────────────
  "split": {
    "oot_months": 6,      # últimos N meses reservados como holdout
    "embargo": True,      # excluir del train las filas cuya ventana de
                          # etiqueta invade el test. Desactivarlo infla
                          # las métricas.
    "n_folds": 5,         # folds de StratifiedGroupKFold (agrupado por CIF)
    "random_state": 42,
  },

  # ── Modelos ─────────────────────────────────────────────────────────
  "models": {
    # `mlp` está definido en el zoo pero fuera de esta lista: añádelo aquí
    # si lo quieres entrenar.
    "enabled": ["logreg", "random_forest", "lightgbm", "xgboost", "catboost"],
    "n_iter": 20,                     # iteraciones de RandomizedSearchCV
    "scoring": "average_precision",   # PR-AUC, no accuracy ni ROC-AUC
    "n_jobs": -1,
    "calibrate": True,
    "stack_top_k": 3,                 # cuántos modelos entran al stacking

    # Red secuencial sobre las series mensuales (GRU).
    "sequence_model": {
        "enabled": True, "hidden_size": 64, "num_layers": 1, "dropout": 0.2,
        "epochs": 30, "batch_size": 256, "lr": 0.001, "patience": 5,
    },
  },

  # ── Explicabilidad ──────────────────────────────────────────────────
  "explain": {
    "shap_sample_size": 2000,
    "top_n_features": 25,
    "counterfactual_features": [      # palancas accionables por retención
        "has_direct_deposit", "digital_login_count_3m_avg",
        "n_products_owned", "bill_pay_payees_active", "fees_charged_6m_sum",
    ],
  },

  # ── Generador sintético (solo si data.source == "synthetic") ────────
  "synthetic": {
    "n_customers": 4000,
    "n_months": 36,
    "start_month": "2022-07-01",
    # Tasa de *intención* de fuga. La detectada por el target compuesto es
    # menor (~la mitad): quien queda anclado como co-titular nunca llega a
    # cumplir la definición. Pasa igual en un banco real.
    "annual_churn_rate": 0.18,
    "seed": 42,
  },
}

In [ ]:
@dataclass
class Config:
    """Configuración del pipeline. Se construye desde el dict de arriba."""

    data: dict[str, Any] = field(default_factory=dict)
    panel: dict[str, Any] = field(default_factory=dict)
    target: dict[str, Any] = field(default_factory=dict)
    ownership_weights: dict[str, Any] = field(default_factory=dict)
    eligible_roles: list[str] = field(default_factory=list)
    split: dict[str, Any] = field(default_factory=dict)
    models: dict[str, Any] = field(default_factory=dict)
    explain: dict[str, Any] = field(default_factory=dict)
    synthetic: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_dict(cls, raw: dict) -> "Config":
        known = {f.name for f in cls.__dataclass_fields__.values()}
        return cls(**{k: v for k, v in raw.items() if k in known})

    @property
    def is_synthetic(self) -> bool:
        return self.data.get("source", "synthetic") == "synthetic"

    @property
    def label_offset_months(self) -> int:
        """Meses entre el as-of date y el inicio de la ventana de performance."""
        return int(self.panel.get("gap_months", 1)) + 1

    def artifacts_path(self) -> Path:
        p = Path(self.data.get("artifacts_dir", "artifacts"))
        p.mkdir(parents=True, exist_ok=True)
        return p

    def processed_path(self) -> Path:
        p = Path(self.data.get("processed_dir", "data/processed"))
        p.mkdir(parents=True, exist_ok=True)
        return p


cfg = Config.from_dict(CONFIG)
print(f"Fuente de datos: {cfg.data['source']}")
print(f"Ventanas: feature={cfg.panel['feature_window_months']}m  "
      f"gap={cfg.panel['gap_months']}m  "
      f"performance={cfg.panel['performance_window_months']}m")

## 2 · Contrato de datos

La frontera entre el agente SQL y el pipeline: si un extracto cumple este contrato,
todo lo demás corre sin tocar nada.

- `required` — sin estas columnas el pipeline no puede correr.
- `optional` — se usan si existen; si no, las features derivadas se omiten.
- `keys` — contra ellas se comprueban duplicados en la validación.

In [ ]:
TABLE_SCHEMAS: dict[str, dict[str, Any]] = {
    "party": {
        "keys": ["cif_id", "as_of_month"],
        "required": ["cif_id", "as_of_month", "entity_type", "customer_since_date"],
        "optional": [
            "age", "tenure_months", "residency_status", "country_of_residence",
            "state", "zip3", "primary_branch_id", "segment", "employee_flag",
            "kyc_risk_rating", "naics", "preferred_language", "deceased_flag",
            # `deceased_date` es preferible a la bandera: dice desde cuándo, y
            # es lo único utilizable si el maestro no tiene dimensión temporal.
            "deceased_date", "status", "merged_into_cif", "household_id",
        ],
        "dates": ["as_of_month", "customer_since_date", "deceased_date"],
    },
    "relationships": {
        "keys": ["cif_id", "account_id", "relationship_role"],
        "required": [
            "cif_id", "account_id", "product_type", "relationship_role",
            "relationship_start_date",
        ],
        "optional": ["relationship_end_date", "n_owners"],
        "dates": ["relationship_start_date", "relationship_end_date"],
    },
    "deposits": {
        "keys": ["account_id", "as_of_month"],
        "required": ["account_id", "as_of_month", "product_family", "eom_balance"],
        "optional": [
            "product_code", "open_date", "close_date", "close_reason_code",
            "account_status", "avg_daily_balance_month", "min_balance_month",
            "max_balance_month", "interest_rate_paid", "maturity_date",
            "term_months", "auto_renew_flag", "original_deposit_amount",
            "overdraft_days_count", "nsf_count_month", "fees_charged_month",
            "statement_delivery", "dormant_flag",
        ],
        "dates": ["as_of_month", "open_date", "close_date", "maturity_date"],
    },
    "loans": {
        "keys": ["loan_id", "as_of_month"],
        "required": ["loan_id", "as_of_month", "loan_type", "current_balance"],
        "optional": [
            "origination_date", "original_amount", "original_term_months",
            "maturity_date", "scheduled_payment", "interest_rate", "rate_type",
            "remaining_term_months", "months_to_maturity", "extra_principal_paid_month",
            "delinquency_bucket", "days_past_due", "times_30dpd_last_12m",
            "status", "payoff_date", "credit_limit", "utilization", "ltv",
        ],
        "dates": ["as_of_month", "origination_date", "maturity_date", "payoff_date"],
    },
    "transactions": {
        # Formato largo: una fila por (cuenta, mes, categoría)
        "keys": ["account_id", "as_of_month", "txn_category"],
        "required": ["account_id", "as_of_month", "txn_category", "txn_count"],
        "optional": [
            "total_amount_debit", "total_amount_credit",
            "last_customer_initiated_txn_date", "days_since_last_customer_txn",
            "is_direct_deposit", "competitor_transfer_flag",
        ],
        "dates": ["as_of_month", "last_customer_initiated_txn_date"],
    },
    "digital": {
        "keys": ["cif_id", "as_of_month"],
        "required": ["cif_id", "as_of_month"],
        "optional": [
            "online_enrolled_flag", "mobile_enrolled_flag", "login_count_web_month",
            "login_count_mobile_month", "last_login_date", "alerts_enrolled_flag",
            "estatement_flag", "debit_card_active_flag", "call_center_calls_month",
            "branch_visits_month", "complaints_open_month", "disputes_month",
            "bill_pay_payees_active",
        ],
        "dates": ["as_of_month", "last_login_date"],
    },
    "events": {
        "keys": ["cif_id", "account_id", "event_date", "event_type"],
        "required": ["cif_id", "event_date", "event_type"],
        "optional": ["account_id", "event_detail", "reason_code"],
        "dates": ["event_date"],
    },
}


# Categorías de transacción que el pipeline sabe explotar. Cualquier otra
# categoría presente en los datos se agrega igual, solo que sin feature
# de dominio dedicada.
KNOWN_TXN_CATEGORIES = [
    "ach_credit_in", "ach_debit_out", "zelle_in", "zelle_out",
    "wire_in", "wire_out", "pos_signature", "pos_pin", "atm_own",
    "atm_foreign", "check_written", "branch_deposit", "mobile_deposit",
    "bill_pay", "internal_transfer",
]

## 3 · Generador sintético

Datos simulados con dinámica causal —perder la nómina baja el saldo, el saldo bajo
precede a la inactividad— para validar el pipeline de punta a punta antes de tener
los extractos reales. Con `data.source = "drive"` nada de esto se ejecuta.

In [ ]:
DEPOSIT_PRODUCTS = ["checking", "savings", "mma", "cd"]
LOAN_TYPES = ["mortgage", "auto", "personal", "heloc", "credit_card"]
INVOLUNTARY_CODES = ["DECEASED", "AML_EXIT", "FRAUD", "CHARGE_OFF"]


def generate(
    n_customers: int = 3000,
    n_months: int = 36,
    start_month: str = "2022-07-01",
    annual_churn_rate: float = 0.09,
    seed: int = 42,
) -> dict[str, pd.DataFrame]:
    """Genera las 7 tablas. Devuelve un dict con las mismas llaves que
    `Config.data['files']`."""
    rng = np.random.default_rng(seed)
    months = pd.date_range(start=start_month, periods=n_months, freq="MS")

    customers = _make_customers(rng, n_customers, months, annual_churn_rate)
    accounts, relationships = _make_accounts(rng, customers, months)

    party = _make_party(customers, months)
    deposits, txns = _make_deposit_history(rng, customers, accounts, months)
    loans = _make_loan_history(rng, customers, accounts, months)
    digital = _make_digital(rng, customers, months)
    events = _make_events(customers, accounts, deposits, loans)
    # Las relaciones se cierran cuando se cierra la cuenta subyacente. Sin
    # esto, un CIF que cerró todo seguiría apareciendo como vinculado y la
    # rama "sin relaciones" del target nunca dispararía.
    relationships = _close_relationships(relationships, deposits, loans)

    return {
        "party": party,
        "relationships": relationships,
        "deposits": deposits,
        "loans": loans,
        "transactions": txns,
        "digital": digital,
        "events": events,
    }


# ── Población ───────────────────────────────────────────────────────────

def _make_customers(rng, n, months, annual_churn_rate) -> pd.DataFrame:
    n_months = len(months)
    cif_ids = [f"CIF{i:07d}" for i in range(n)]

    entity_type = rng.choice(["personal", "business"], n, p=[0.85, 0.15])
    tenure_start = rng.integers(6, 180, n)  # meses de antigüedad al inicio
    segment = rng.choice(["mass", "affluent", "premier", "small_business"], n,
                         p=[0.62, 0.22, 0.06, 0.10])
    residency = rng.choice(["resident", "non_resident"], n, p=[0.82, 0.18])

    # ── Riesgo latente: combina factores estructurales ──
    base_balance = np.exp(rng.normal(8.6, 1.35, n))  # lognormal, mediana ~5.5k
    has_dd_initial = rng.random(n) < np.where(entity_type == "personal", 0.58, 0.25)
    n_products = rng.integers(1, 5, n)
    digital_affinity = rng.beta(2.4, 2.0, n)

    # Menos productos, sin nómina, poco digital y poca antigüedad => más riesgo
    risk = (
        -0.40 * has_dd_initial
        - 0.22 * np.log1p(n_products)
        - 0.30 * digital_affinity
        - 0.12 * np.log1p(tenure_start / 12)
        - 0.10 * np.log1p(base_balance) / 5
        + 0.25 * (residency == "non_resident")
        + rng.normal(0, 0.45, n)
    )
    # Calibrar la tasa de churn del periodo completo al objetivo anual
    horizon_years = n_months / 12
    target_rate = 1 - (1 - annual_churn_rate) ** horizon_years
    thresh = np.quantile(risk, 1 - target_rate)
    will_churn = risk > thresh

    # Mes del evento: uniforme sobre la ventana, dejando margen para señales
    # Margen al final: el evento debe caer dentro de la ventana observada
    churn_month_idx = np.where(
        will_churn,
        rng.integers(10, max(11, n_months - 2), n),
        -1,
    )

    # ~12% de las salidas son involuntarias (se excluyen del modelo)
    involuntary = will_churn & (rng.random(n) < 0.12)
    reason = np.where(
        involuntary,
        rng.choice(INVOLUNTARY_CODES, n),
        np.where(will_churn, "CUSTOMER_REQUEST", None),
    )
    # Solo ~45% de los que se van cierran formalmente: el resto es
    # attrition silenciosa (la trampa que el target compuesto debe capturar)
    formal_close = will_churn & (rng.random(n) < 0.45)

    return pd.DataFrame({
        "cif_id": cif_ids,
        "entity_type": entity_type,
        "tenure_start_months": tenure_start,
        "segment": segment,
        "residency_status": residency,
        "base_balance": base_balance,
        "has_dd_initial": has_dd_initial,
        "n_products": n_products,
        "digital_affinity": digital_affinity,
        "will_churn": will_churn,
        "churn_month_idx": churn_month_idx,
        "involuntary": involuntary,
        "close_reason_code": reason,
        "formal_close": formal_close,
        # Adelanto con que cada señal se degrada antes del evento
        "dd_loss_lead": rng.integers(3, 8, n),
        "balance_decay_lead": rng.integers(4, 10, n),
        "login_decay_lead": rng.integers(2, 7, n),
        "state": rng.choice(["FL", "TX", "NY", "NJ", "CA"], n,
                            p=[0.55, 0.13, 0.12, 0.10, 0.10]),
        "employee_flag": rng.random(n) < 0.008,
    })


def _close_relationships(relationships: pd.DataFrame, deposits: pd.DataFrame,
                         loans: pd.DataFrame) -> pd.DataFrame:
    """Propaga el cierre de cuentas y el payoff de préstamos a la vigencia de
    las relaciones CIF↔cuenta."""
    ends = []
    if len(deposits):
        d = deposits.dropna(subset=["close_date"])
        if len(d):
            ends.append(d.groupby("account_id")["close_date"].min())
    if len(loans) and "payoff_date" in loans.columns:
        lo = loans.dropna(subset=["payoff_date"])
        if len(lo):
            ends.append(lo.groupby("loan_id")["payoff_date"].min()
                        .rename_axis("account_id"))
    if not ends:
        return relationships

    end_map = pd.concat(ends).groupby(level=0).min()
    rel = relationships.copy()
    mapped = rel["account_id"].map(end_map)
    rel["relationship_end_date"] = rel["relationship_end_date"].fillna(mapped)
    return rel


def _make_party(customers: pd.DataFrame, months) -> pd.DataFrame:
    """Q1: una fila por CIF por mes."""
    # El alta es un atributo inmutable del cliente, así que se ancla al primer
    # mes del panel y no se recalcula en cada iteración. Derivarla de `m` la
    # hacía avanzar un mes por cada mes del panel: el mismo CIF aparecía dado
    # de alta en 2017 al principio del panel y en 2020 al final, lo que
    # contradice su propio `tenure_months` y la inutiliza como ancla temporal.
    customer_since = pd.Series(
        months[0] - pd.to_timedelta(
            customers["tenure_start_months"].to_numpy() * 30, unit="D"),
        index=customers.index,
    )
    rows = []
    for i, m in enumerate(months):
        rows.append(pd.DataFrame({
            "cif_id": customers["cif_id"],
            "as_of_month": m,
            "entity_type": customers["entity_type"],
            "customer_since_date": customer_since,
            # Derivada de la fecha de alta, no contada aparte: con el mes de
            # 30 días de la línea anterior, `tenure_start + i` se separaba
            # hasta dos meses de lo que dice `customer_since_date`.
            "tenure_months": ((m.year - customer_since.dt.year) * 12
                              + (m.month - customer_since.dt.month)).clip(lower=0),
            "segment": customers["segment"],
            "residency_status": customers["residency_status"],
            "state": customers["state"],
            "employee_flag": customers["employee_flag"],
            "deceased_flag": (customers["close_reason_code"] == "DECEASED")
                             & (i >= customers["churn_month_idx"]),
            "kyc_risk_rating": np.where(
                customers["residency_status"] == "non_resident", "medium", "low"
            ),
            "status": "active",
        }))
    return pd.concat(rows, ignore_index=True)


# ── Cuentas y relaciones ────────────────────────────────────────────────

def _make_accounts(rng, customers: pd.DataFrame, months):
    """Genera cuentas de depósito y préstamos, con roles de CIF realistas
    (incluyendo joint y authorized signer)."""
    acct_rows, rel_rows = [], []
    n = len(customers)
    acct_seq = 0
    start = months[0]

    for idx in range(n):
        cif = customers.at[idx, "cif_id"]
        n_dep = max(1, min(4, int(customers.at[idx, "n_products"])))
        share = rng.dirichlet(np.ones(n_dep))

        for j in range(n_dep):
            acct_seq += 1
            acct_id = f"ACC{acct_seq:08d}"
            product = "checking" if j == 0 else rng.choice(
                DEPOSIT_PRODUCTS[1:], p=[0.45, 0.25, 0.30]
            )
            is_joint = rng.random() < 0.22
            n_owners = 2 if is_joint else 1

            acct_rows.append({
                "account_id": acct_id,
                "owner_cif": cif,
                "product_family": product,
                "balance_share": share[j],
                "n_owners": n_owners,
                "open_offset": -int(rng.integers(1, 200)),
                "term_months": int(rng.choice([6, 12, 24, 36, 60])) if product == "cd" else None,
                "rate": float(rng.uniform(0.001, 0.045)) if product in ("cd", "mma", "savings")
                        else float(rng.uniform(0.0, 0.002)),
            })
            rel_rows.append({
                "cif_id": cif, "account_id": acct_id, "product_type": "deposit",
                "relationship_role": "joint" if is_joint else "primary",
                "relationship_start_date": start + pd.DateOffset(
                    months=acct_rows[-1]["open_offset"]
                ),
                "relationship_end_date": pd.NaT, "n_owners": n_owners,
            })
            # Co-titular en cuentas joint: otro CIF de la población
            if is_joint:
                other = customers.at[int(rng.integers(0, n)), "cif_id"]
                if other != cif:
                    rel_rows.append({
                        "cif_id": other, "account_id": acct_id,
                        "product_type": "deposit", "relationship_role": "joint",
                        "relationship_start_date": rel_rows[-1]["relationship_start_date"],
                        "relationship_end_date": pd.NaT, "n_owners": n_owners,
                    })
            # Authorized signer: sin peso económico, pero ancla relacional
            if rng.random() < 0.08:
                other = customers.at[int(rng.integers(0, n)), "cif_id"]
                if other != cif:
                    rel_rows.append({
                        "cif_id": other, "account_id": acct_id,
                        "product_type": "deposit",
                        "relationship_role": "authorized_signer",
                        "relationship_start_date": rel_rows[-1]["relationship_start_date"],
                        "relationship_end_date": pd.NaT, "n_owners": n_owners,
                    })

        # Préstamo (~35% de los clientes)
        if rng.random() < 0.35:
            acct_seq += 1
            loan_id = f"LN{acct_seq:08d}"
            ltype = str(rng.choice(LOAN_TYPES, p=[0.18, 0.30, 0.24, 0.10, 0.18]))
            term = int(rng.choice([36, 48, 60, 120, 360]))
            amt = float(np.exp(rng.normal(10.2, 0.9)))
            elapsed = int(rng.integers(1, max(2, min(term - 1, 60))))
            acct_rows.append({
                "account_id": loan_id, "owner_cif": cif, "product_family": "loan",
                "loan_type": ltype, "original_amount": amt,
                "original_term_months": term, "elapsed_at_start": elapsed,
                "rate": float(rng.uniform(0.03, 0.19)),
                "balance_share": 0.0, "n_owners": 1, "open_offset": -elapsed,
            })
            rel_rows.append({
                "cif_id": cif, "account_id": loan_id, "product_type": "loan",
                "relationship_role": "borrower",
                "relationship_start_date": start + pd.DateOffset(months=-elapsed),
                "relationship_end_date": pd.NaT, "n_owners": 1,
            })

    return pd.DataFrame(acct_rows), pd.DataFrame(rel_rows)


# ── Historia de depósitos y transacciones ───────────────────────────────

def _make_deposit_history(rng, customers: pd.DataFrame, accounts: pd.DataFrame, months):
    """Q3 (snapshot mensual de depósitos) y Q5 (transacciones en formato largo)."""
    cust = customers.set_index("cif_id")
    dep_accts = accounts[accounts["product_family"] != "loan"].reset_index(drop=True)
    n_months = len(months)

    dep_rows, txn_rows = [], []

    for r in dep_accts.itertuples(index=False):
        c = cust.loc[r.owner_cif]
        churn_i = int(c.churn_month_idx)
        involuntary = bool(c.involuntary)
        base = float(c.base_balance) * float(r.balance_share)
        trend = rng.normal(0.004, 0.012)          # deriva mensual del saldo
        noise = max(0.04, rng.gamma(2.0, 0.035))  # volatilidad idiosincrática
        open_date = months[0] + pd.DateOffset(months=int(r.open_offset))
        maturity = (open_date + pd.DateOffset(months=int(r.term_months))
                    if r.product_family == "cd" and r.term_months else pd.NaT)

        closed = False
        days_since_txn = int(rng.integers(0, 20))

        for i, m in enumerate(months):
            if closed:
                break

            # ── Trayectoria del saldo ──
            level = base * (1 + trend) ** i
            if churn_i >= 0:
                lead = int(c.balance_decay_lead)
                if i >= churn_i - lead:
                    # Decaimiento exponencial hacia ~0 al llegar al evento
                    progress = np.clip((i - (churn_i - lead)) / max(lead, 1), 0, 2.0)
                    level *= float(np.exp(-3.6 * progress))
            bal = max(0.0, level * float(rng.normal(1.0, noise)))

            # ── Nómina / direct deposit ──
            has_dd = bool(c.has_dd_initial)
            if has_dd and churn_i >= 0 and i >= churn_i - int(c.dd_loss_lead):
                has_dd = False

            # ── Actividad transaccional ──
            # Llega a cero ~3 meses ANTES del evento: así el contador de
            # inactividad alcanza a acumularse y el target compuesto dispara
            # cerca del mes real de fuga, no cinco meses después.
            activity = 1.0
            if churn_i >= 0:
                lead = int(c.balance_decay_lead)
                if i >= churn_i - lead:
                    activity = float(np.clip(
                        1 - (i - (churn_i - lead)) / max(lead - 3, 1), 0.0, 1.0
                    ))

            is_primary_dda = r.product_family == "checking"
            intensity = activity * (2.2 if is_primary_dda else 0.5)

            # ── Cierre formal ──
            close_date, reason, status = pd.NaT, None, "open"
            if churn_i >= 0 and i >= churn_i and bool(c.formal_close):
                close_date, reason, status = m, c.close_reason_code, "closed"
                closed = True
            elif bal < 50 and activity < 0.15:
                status = "dormant"

            nsf = int(rng.poisson(0.55 * (1.9 if (churn_i >= 0 and
                      churn_i - 6 <= i < churn_i) else 1.0))) if is_primary_dda else 0
            fees = float(nsf * 35 + (12.0 if bal < 1500 and is_primary_dda else 0.0))

            dep_rows.append({
                "account_id": r.account_id, "as_of_month": m,
                "product_family": r.product_family, "product_code": f"{r.product_family.upper()}01",
                "open_date": open_date, "close_date": close_date,
                "close_reason_code": reason, "account_status": status,
                "eom_balance": round(bal, 2),
                "avg_daily_balance_month": round(bal * float(rng.uniform(0.9, 1.1)), 2),
                "min_balance_month": round(bal * float(rng.uniform(0.55, 0.95)), 2),
                "interest_rate_paid": r.rate,
                "maturity_date": maturity,
                "term_months": r.term_months,
                "auto_renew_flag": bool(rng.random() < 0.6) if r.product_family == "cd" else None,
                "nsf_count_month": nsf,
                "fees_charged_month": round(fees, 2),
                "overdraft_days_count": int(rng.poisson(0.3)) if is_primary_dda else 0,
                "statement_delivery": "e" if float(c.digital_affinity) > 0.45 else "paper",
                "dormant_flag": status == "dormant",
            })

            # ── Transacciones (formato largo, solo filas con actividad) ──
            month_txn_total = 0
            cats = {
                "ach_credit_in": (2.0 + 1.6 * has_dd) * intensity,
                "ach_debit_out": 2.4 * intensity,
                "pos_signature": 9.0 * intensity if is_primary_dda else 0.4,
                "pos_pin": 4.0 * intensity if is_primary_dda else 0.2,
                "atm_own": 1.4 * intensity,
                "atm_foreign": 0.5 * intensity,
                "zelle_out": 1.7 * intensity,
                "zelle_in": 1.1 * intensity,
                "bill_pay": 2.6 * intensity if is_primary_dda else 0.1,
                "mobile_deposit": 1.0 * intensity * float(c.digital_affinity) * 2,
                "internal_transfer": 0.9 * intensity,
                "wire_out": 0.12 * intensity,
            }
            # Fuga en curso: sube la transferencia saliente a otra institución
            competitor_out = 0
            if churn_i >= 0 and churn_i - 6 <= i < churn_i:
                competitor_out = int(rng.poisson(1.8))

            for cat, lam in cats.items():
                cnt = int(rng.poisson(max(lam, 0)))
                if cnt == 0:
                    continue
                month_txn_total += cnt
                debit = cat.endswith(("_out", "_written", "signature", "pin")) or "atm" in cat
                amt = float(cnt * abs(rng.normal(bal * 0.05 + 60, 45)))
                txn_rows.append({
                    "account_id": r.account_id, "as_of_month": m, "txn_category": cat,
                    "txn_count": cnt,
                    "total_amount_debit": round(amt, 2) if debit else 0.0,
                    "total_amount_credit": 0.0 if debit else round(amt, 2),
                    "is_direct_deposit": bool(has_dd and cat == "ach_credit_in"),
                    "competitor_transfer_flag": False,
                    "days_since_last_customer_txn": days_since_txn,
                })

            if competitor_out:
                # Categoría propia para no romper la llave (cuenta, mes, categoría)
                month_txn_total += competitor_out
                txn_rows.append({
                    "account_id": r.account_id, "as_of_month": m,
                    "txn_category": "ach_debit_out_competitor",
                    "txn_count": competitor_out,
                    "total_amount_debit": round(float(competitor_out) * bal * 0.28 + 200, 2),
                    "total_amount_credit": 0.0, "is_direct_deposit": False,
                    "competitor_transfer_flag": True,
                    "days_since_last_customer_txn": days_since_txn,
                })

            days_since_txn = 0 if month_txn_total > 0 else days_since_txn + 30

            if involuntary and churn_i >= 0 and i >= churn_i:
                closed = True

    deposits = pd.DataFrame(dep_rows)
    txns = pd.DataFrame(txn_rows)
    return deposits, txns


def _make_loan_history(rng, customers, accounts, months) -> pd.DataFrame:
    """Q4: snapshot mensual de préstamos, con payoff natural vs. prepago."""
    cust = customers.set_index("cif_id")
    loan_accts = accounts[accounts["product_family"] == "loan"]
    rows = []

    for r in loan_accts.itertuples(index=False):
        c = cust.loc[r.owner_cif]
        churn_i = int(c.churn_month_idx)
        term = int(r.original_term_months)
        amt = float(r.original_amount)
        rate_m = float(r.rate) / 12
        elapsed0 = int(r.elapsed_at_start)
        orig_date = months[0] + pd.DateOffset(months=-elapsed0)
        maturity = orig_date + pd.DateOffset(months=term)
        pmt = amt * rate_m / (1 - (1 + rate_m) ** -term) if rate_m > 0 else amt / term

        # Prepago: más probable si el cliente va de salida
        prepay_i = -1
        if churn_i >= 0 and rng.random() < 0.45:
            prepay_i = max(0, churn_i - int(rng.integers(0, 4)))

        bal = amt * ((1 + rate_m) ** term - (1 + rate_m) ** elapsed0) / \
              ((1 + rate_m) ** term - 1) if rate_m > 0 else amt * (1 - elapsed0 / term)
        bal = max(bal, 0.0)
        status = "active"

        for i, m in enumerate(months):
            k = elapsed0 + i
            if status != "active":
                break
            months_to_maturity = term - k

            if months_to_maturity <= 0:
                status, payoff = "paid_off", m       # maturity natural
                bal = 0.0
            elif i == prepay_i:
                status, payoff = "paid_off", m       # prepago competitivo
                bal = 0.0
            elif c.close_reason_code == "CHARGE_OFF" and churn_i >= 0 and i >= churn_i:
                status, payoff = "charged_off", m
            else:
                payoff = pd.NaT
                interest = bal * rate_m
                bal = max(0.0, bal - max(pmt - interest, 0))

            dpd_risk = 0.05 + (0.25 if churn_i >= 0 and i >= churn_i - 4 else 0)
            dpd = int(rng.choice([0, 30, 60, 90], p=[1 - dpd_risk, dpd_risk * 0.6,
                                                     dpd_risk * 0.28, dpd_risk * 0.12]))

            rows.append({
                "loan_id": r.account_id, "as_of_month": m, "loan_type": r.loan_type,
                "origination_date": orig_date, "original_amount": round(amt, 2),
                "original_term_months": term, "maturity_date": maturity,
                "current_balance": round(bal, 2), "scheduled_payment": round(pmt, 2),
                "interest_rate": r.rate, "rate_type": "fixed",
                "remaining_term_months": max(months_to_maturity, 0),
                "months_to_maturity": max(months_to_maturity, 0),
                "delinquency_bucket": {0: "current", 30: "30", 60: "60", 90: "90+"}[dpd],
                "days_past_due": dpd, "status": status, "payoff_date": payoff,
                "utilization": (round(1 - bal / amt, 4) if amt > 0 else 0.0),
            })

    if not rows:
        return pd.DataFrame(columns=["loan_id", "as_of_month", "loan_type", "current_balance"])
    return pd.DataFrame(rows)


def _make_digital(rng, customers: pd.DataFrame, months) -> pd.DataFrame:
    """Q6: actividad digital y de contacto por CIF-mes."""
    rows = []
    n = len(customers)
    aff = customers["digital_affinity"].to_numpy()
    churn_idx = customers["churn_month_idx"].to_numpy()
    login_lead = customers["login_decay_lead"].to_numpy()
    enrolled = rng.random(n) < (0.35 + 0.55 * aff)

    for i, m in enumerate(months):
        decay = np.ones(n)
        active_churn = churn_idx >= 0
        start_decay = churn_idx - login_lead
        mask = active_churn & (i >= start_decay)
        decay[mask] = np.clip(
            1 - (i - start_decay[mask]) / np.maximum(login_lead[mask], 1), 0.0, 1.0
        )

        web = rng.poisson(np.maximum(aff * 6 * decay * enrolled, 0.01))
        mob = rng.poisson(np.maximum(aff * 11 * decay * enrolled, 0.01))
        complaint_rate = np.where(mask, 0.10, 0.02)

        rows.append(pd.DataFrame({
            "cif_id": customers["cif_id"], "as_of_month": m,
            "online_enrolled_flag": enrolled,
            "mobile_enrolled_flag": enrolled & (aff > 0.35),
            "login_count_web_month": web, "login_count_mobile_month": mob,
            "alerts_enrolled_flag": enrolled & (aff > 0.55),
            "estatement_flag": aff > 0.45,
            "debit_card_active_flag": rng.random(n) < 0.85,
            "call_center_calls_month": rng.poisson(np.where(mask, 0.9, 0.25)),
            "branch_visits_month": rng.poisson(np.maximum((1 - aff) * 1.4 * decay, 0.01)),
            "complaints_open_month": rng.binomial(1, complaint_rate),
            "disputes_month": rng.binomial(1, 0.02),
            "bill_pay_payees_active": rng.poisson(np.maximum(aff * 4 * decay, 0.01)),
        }))
    return pd.concat(rows, ignore_index=True)


def _make_events(customers, accounts, deposits, loans) -> pd.DataFrame:
    """Q7: log de eventos de producto."""
    rows = []

    opens = accounts[["account_id", "owner_cif"]].copy()
    first = deposits.groupby("account_id", as_index=False)["open_date"].first()
    opens = opens.merge(first, on="account_id", how="inner")
    for r in opens.itertuples(index=False):
        rows.append({"cif_id": r.owner_cif, "account_id": r.account_id,
                     "event_date": r.open_date, "event_type": "account_open",
                     "reason_code": None})

    closes = deposits[deposits["close_date"].notna()]
    owner = accounts.set_index("account_id")["owner_cif"]
    for r in closes.itertuples(index=False):
        rows.append({"cif_id": owner.get(r.account_id), "account_id": r.account_id,
                     "event_date": r.close_date, "event_type": "account_close",
                     "reason_code": r.close_reason_code})

    if len(loans):
        po = loans[loans["payoff_date"].notna()]
        for r in po.itertuples(index=False):
            rows.append({"cif_id": owner.get(r.loan_id), "account_id": r.loan_id,
                         "event_date": r.payoff_date, "event_type": "loan_payoff",
                         "reason_code": r.status})

    ev = pd.DataFrame(rows)
    return ev.dropna(subset=["cif_id"]).reset_index(drop=True)

## 4 · Carga, despivote y validación

`load_tables` es el único punto de entrada: el resto del pipeline no sabe —ni le
importa— si los datos vienen de Drive o del generador.

Aquí vive también el soporte del **formato ancho**. Bajar el panel por ODBC a Excel
es inviable en formato largo (el tope son 1.048.576 filas, y `transactions` sola son
~25M con 50.000 CIFs), así que los meses se piden como columnas. No hay que declarar
nada: si una tabla trae el mes en el nombre de columna y no trae `as_of_month`, se
despivota al cargar.

In [ ]:
# ── Google Drive ────────────────────────────────────────────────────────

def in_colab() -> bool:
    return "google.colab" in sys.modules


def mount_drive(mount_point: str = "/content/drive", force: bool = False) -> bool:
    """Monta Google Drive si estamos en Colab. Fuera de Colab no hace nada
    (permite correr el mismo código local sin tocarlo)."""
    if not in_colab():
        print("ℹ️  Fuera de Colab: se omite el montaje de Drive.")
        return False
    if Path(mount_point, "MyDrive").exists() and not force:
        print(f"✓ Drive ya montado en {mount_point}")
        return True
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount(mount_point, force_remount=force)
    print(f"✓ Drive montado en {mount_point}")
    return True


# ── Carga ───────────────────────────────────────────────────────────────

def load_tables(cfg: Config, verbose: bool = True) -> dict[str, pd.DataFrame]:
    """Devuelve las 7 tablas como dict. Las tablas de `data.optional` que
    falten se devuelven vacías, para poder empezar solo con Q1–Q3."""
    if cfg.is_synthetic:
        s = cfg.synthetic
        if verbose:
            print(f"⚙️  Generando datos sintéticos "
                  f"({s.get('n_customers')} clientes × {s.get('n_months')} meses)…")
        tables = generate(
            n_customers=int(s.get("n_customers", 3000)),
            n_months=int(s.get("n_months", 36)),
            start_month=str(s.get("start_month", "2022-07-01")),
            annual_churn_rate=float(s.get("annual_churn_rate", 0.09)),
            seed=int(s.get("seed", 42)),
        )
    else:
        tables = _load_from_disk(cfg, verbose=verbose)

    tables = {k: unpivot_month_columns(k, df, verbose=verbose)
              for k, df in tables.items()}
    if "transactions" in tables:
        tables["transactions"] = unpivot_txn_categories(
            tables["transactions"], verbose=verbose)
    tables = {k: _coerce_dtypes(k, df) for k, df in tables.items()}
    tables = _expand_static_party(tables, verbose=verbose)
    if verbose:
        _print_summary(tables)
    return tables


# ── Formato ancho (una columna por mes) ─────────────────────────────────
#
# Bajar el panel en formato largo por ODBC a Excel es inviable en cuanto hay
# decenas de miles de clientes: el tope de Excel son 1.048.576 filas y la
# tabla de transacciones sola se pasa por un orden de magnitud. La salida es
# pivotar los meses a columnas — el tope de columnas es 16.384, que sobra.
#
# El pipeline acepta las dos orientaciones y no hay que declarar cuál es: si
# una tabla trae columnas con un mes en el nombre y no trae `as_of_month`,
# se despivota aquí y el resto del código no se entera.
#
# Nombres reconocidos, con el mes al final:
#     eom_balance_2024_07   eom_balance_2024-07   eom_balance_202407
#
_MONTH_SUFFIX = re.compile(
    r"^(?P<metric>.+?)[_\-](?P<year>20\d{2})[_\-]?(?P<month>0[1-9]|1[0-2])$"
)


def _restore_numeric(df: pd.DataFrame, skip: tuple[str, ...] = ()) -> pd.DataFrame:
    """Devuelve su tipo numérico a las columnas que salen del despivote.

    Fundir todas las métricas en una sola columna de valores obliga a un dtype
    común, y basta una métrica de texto (`account_status`) para que todas
    acaben como `object`. Con ese dtype, `select_dtypes(number)` deja de verlas
    y el relleno de nulos del panel no las alcanza. Se reconvierte solo cuando
    no se pierde ningún valor por el camino.
    """
    for c in df.columns:
        if c in skip or not pd.api.types.is_object_dtype(df[c]):
            continue
        conv = pd.to_numeric(df[c], errors="coerce")
        if conv.notna().sum() == df[c].notna().sum():
            df[c] = conv
    return df


def _split_month_columns(df: pd.DataFrame) -> dict[str, tuple[str, str]]:
    """{columna: (métrica, 'YYYY-MM')} para las columnas con mes en el nombre."""
    out = {}
    for c in df.columns:
        m = _MONTH_SUFFIX.match(str(c))
        if m:
            out[c] = (m.group("metric"), f"{m.group('year')}-{m.group('month')}")
    return out


def unpivot_month_columns(table: str, df: pd.DataFrame,
                          verbose: bool = True) -> pd.DataFrame:
    """Convierte un extracto ancho (un mes por columna) a formato largo.

    No toca nada si la tabla ya viene larga o no tiene columnas de mes.
    """
    if df is None or df.empty or "as_of_month" in df.columns:
        return df

    month_cols = _split_month_columns(df)
    if not month_cols:
        return df

    id_cols = [c for c in df.columns if c not in month_cols]
    # El pivote va sobre un índice de fila sintético, no sobre las columnas
    # identificadoras: `pivot_table` descarta las filas con NaN en el índice, y
    # las estáticas nulas son la norma (`close_date` en una cuenta abierta,
    # `maturity_date` en una cuenta corriente). Con las identificadoras en el
    # índice se perdería la mayor parte del extracto en silencio.
    src = df.reset_index(drop=True)
    src["_row"] = range(len(src))

    long = src.melt(id_vars=["_row"], value_vars=list(month_cols),
                    var_name="_col", value_name="_val")
    meta = long["_col"].map(month_cols)
    long["_metric"] = meta.str[0]
    long["as_of_month"] = pd.to_datetime(meta.str[1], format="%Y-%m")
    long = long.drop(columns="_col")

    pivoted = (long.pivot_table(index=["_row", "as_of_month"], columns="_metric",
                               values="_val", aggfunc="first")
               .reset_index().rename_axis(columns=None))
    pivoted = _restore_numeric(pivoted, skip=("_row", "as_of_month"))
    wide = (src[id_cols + ["_row"]].merge(pivoted, on="_row", how="inner")
            .drop(columns="_row"))

    if verbose:
        n_metrics = len({m for m, _ in month_cols.values()})
        n_months = len({d for _, d in month_cols.values()})
        print(f"   ↔️  {table:14s} ancho → largo: {len(df):,} filas × "
              f"{len(month_cols)} columnas de mes "
              f"({n_metrics} métricas × {n_months} meses) → {len(wide):,} filas")
    return wide


# Métricas de la tabla de transacciones que pueden venir con la categoría
# pegada al nombre de la columna.
_TXN_VALUE_COLS = ("txn_count", "total_amount_debit", "total_amount_credit")


def unpivot_txn_categories(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Despivota la categoría de transacción cuando viene en el nombre.

    `transactions` es, con diferencia, la tabla más grande: con 50.000 CIFs
    son ~25M de filas en formato largo. Pivotar solo el mes no alcanza —quedan
    `cuenta × categoría`, todavía por encima del millón—, así que la categoría
    también acaba en el nombre de la columna:

        txn_count_zelle_out_2024_07

    El despivote del mes deja `txn_count_zelle_out`; esta función lo parte en
    `txn_category='zelle_out'` y `txn_count`. Las filas sin actividad se
    descartan: materializar cuenta × mes × categoría reproduciría en memoria
    justo la explosión que el formato ancho evita.
    """
    if df is None or df.empty or "txn_category" in df.columns:
        return df

    # La categoría se deduce quitando el prefijo de la métrica, no contrastando
    # contra una lista cerrada: cualquier categoría que el core emita y no esté
    # en KNOWN_TXN_CATEGORIES se perdería en silencio, y son justo las
    # específicas del banco (p. ej. `ach_debit_out_competitor`) las que más
    # señal traen.
    mapping: dict[str, tuple[str, str]] = {}
    for c in df.columns:
        name = str(c)
        for metric in _TXN_VALUE_COLS:
            if name.startswith(f"{metric}_"):
                mapping[c] = (metric, name[len(metric) + 1:])
                break
    if not mapping:
        return df

    unknown = {cat for _, cat in mapping.values()} - set(KNOWN_TXN_CATEGORIES)
    if unknown and verbose:
        print(f"   ℹ️  categorías fuera del catálogo, se cargan igual: "
              f"{sorted(unknown)}")

    id_cols = [c for c in df.columns if c not in mapping]
    # Mismo cuidado que en el despivote del mes: índice de fila sintético para
    # que las identificadoras nulas no se lleven la fila por delante.
    src = df.reset_index(drop=True)
    src["_row"] = range(len(src))

    long = src.melt(id_vars=["_row"], value_vars=list(mapping),
                    var_name="_col", value_name="_val")
    meta = long["_col"].map(mapping)
    long["_metric"] = meta.str[0]
    long["txn_category"] = meta.str[1]
    long = long.drop(columns="_col")

    pivoted = (long.pivot_table(index=["_row", "txn_category"], columns="_metric",
                               values="_val", aggfunc="first")
               .reset_index().rename_axis(columns=None))
    out = (src[id_cols + ["_row"]].merge(pivoted, on="_row", how="inner")
           .drop(columns="_row"))
    if "txn_count" in out.columns:
        out = out[out["txn_count"].fillna(0) > 0].reset_index(drop=True)

    if verbose:
        n_cat = len({c for _, c in mapping.values()})
        print(f"   ↔️  transactions   categoría en columna → filas: "
              f"{n_cat} categorías → {len(out):,} filas con actividad")
    return out


def _expand_static_party(tables: dict[str, pd.DataFrame],
                         verbose: bool = True) -> dict[str, pd.DataFrame]:
    """Expande un maestro de clientes sin dimensión temporal a CIF×mes.

    Con CIFs que no se reutilizan, pedir una fila por cliente y por mes
    multiplica el extracto por el número de meses sin aportar nada: los
    atributos del maestro no cambian. La única excepción es la antigüedad,
    que aquí se recalcula mes a mes desde `customer_since_date`.

    La rejilla de meses sale de `deposits`, que es la tabla que define el
    calendario del panel.
    """
    party = tables.get("party")
    if party is None or party.empty or "as_of_month" in party.columns:
        return tables

    deposits = tables.get("deposits")
    if deposits is None or deposits.empty or "as_of_month" not in deposits.columns:
        raise ValueError(
            "El maestro de clientes no trae `as_of_month` y no hay tabla de "
            "depósitos de la que sacar el calendario de meses.\n"
            "  → O bajas `party` con una fila por CIF y por mes, o te aseguras "
            "de que `deposits` traiga `as_of_month`."
        )

    months = pd.DatetimeIndex(sorted(deposits["as_of_month"].dropna().unique()))
    grid = party.merge(pd.DataFrame({"as_of_month": months}), how="cross")

    # No inventar historia anterior al alta del cliente
    if "customer_since_date" in grid.columns:
        since = pd.to_datetime(grid["customer_since_date"], errors="coerce")
        grid = grid[since.isna() | (since <= grid["as_of_month"])]
        since = pd.to_datetime(grid["customer_since_date"], errors="coerce")
        # La antigüedad del maestro es la de hoy; mes a mes hay que recalcularla
        grid["tenure_months"] = (
            (grid["as_of_month"].dt.year - since.dt.year) * 12
            + (grid["as_of_month"].dt.month - since.dt.month)
        ).clip(lower=0)

    for col in ("cif_close_date", "closed_date", "relationship_end_date"):
        if col in grid.columns:
            end = pd.to_datetime(grid[col], errors="coerce")
            grid = grid[end.isna() | (end > grid["as_of_month"])]
            break

    grid = grid.reset_index(drop=True)
    if "deceased_flag" in grid.columns and "deceased_date" not in grid.columns:
        print("   ⚠️  El maestro trae `deceased_flag` pero no `deceased_date`. "
              "Una bandera sin fecha marca al cliente como fallecido durante "
              "toda su historia y lo saca del panel entero, no desde el "
              "evento. Pide `deceased_date` en el extracto.")
    if verbose:
        print(f"   ↔️  party          maestro estático → panel: "
              f"{len(party):,} CIFs × {len(months)} meses → {len(grid):,} filas")
    tables = dict(tables)
    tables["party"] = grid
    return tables


def _load_from_disk(cfg: Config, verbose: bool = True) -> dict[str, pd.DataFrame]:
    root = Path(cfg.data["drive_dir"])
    fmt = cfg.data.get("file_format", "csv")
    optional = set(cfg.data.get("optional", []))

    if not root.exists():
        raise FileNotFoundError(
            f"No existe la carpeta de datos: {root}\n"
            "  → ¿Montaste Drive? Corre `mount_drive()` primero.\n"
            "  → ¿La ruta de la celda de configuración (data.drive_dir) es la correcta?"
        )

    out: dict[str, pd.DataFrame] = {}
    for key, stem in cfg.data["files"].items():
        path = root / f"{stem}.{fmt}"
        if not path.exists():
            # Tolerar .csv.gz y .parquet indistintamente
            alts = list(root.glob(f"{stem}.*"))
            path = alts[0] if alts else path
        if not path.exists():
            if key in optional:
                if verbose:
                    print(f"⚠️  {key:14s} no encontrada ({stem}.{fmt}) — se omite")
                out[key] = pd.DataFrame()
                continue
            raise FileNotFoundError(
                f"Falta la tabla obligatoria '{key}': {path}\n"
                f"  Archivos presentes: {[p.name for p in root.iterdir()][:20]}"
            )
        out[key] = _read(path)
        if verbose:
            print(f"✓ {key:14s} {len(out[key]):>9,} filas  ← {path.name}")
    return out


def _read(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffixes[-2:] == [".csv", ".gz"] or path.suffix == ".gz":
        return pd.read_csv(path, compression="gzip", low_memory=False)
    return pd.read_csv(path, low_memory=False)


def _coerce_dtypes(table: str, df: pd.DataFrame) -> pd.DataFrame:
    """Convierte columnas de fecha y normaliza as_of_month a inicio de mes."""
    if df.empty:
        return df
    schema = TABLE_SCHEMAS.get(table, {})
    for col in schema.get("dates", []):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    if "as_of_month" in df.columns:
        df["as_of_month"] = df["as_of_month"].dt.to_period("M").dt.to_timestamp()
    return df


# ── Validación y perfilado ──────────────────────────────────────────────

def validate(tables: dict[str, pd.DataFrame], strict: bool = False) -> dict[str, list[str]]:
    """Verifica cada tabla contra el contrato de `TABLE_SCHEMAS`.

    Devuelve {tabla: [problemas]}. Con strict=True lanza excepción si hay
    columnas obligatorias faltantes.
    """
    problems: dict[str, list[str]] = {}

    for name, schema in TABLE_SCHEMAS.items():
        df = tables.get(name)
        if df is None or df.empty:
            continue
        issues = []

        missing = [c for c in schema["required"] if c not in df.columns]
        if missing:
            issues.append(f"faltan columnas obligatorias: {missing}")

        keys = [k for k in schema["keys"] if k in df.columns]
        if keys and len(keys) == len(schema["keys"]):
            dupes = int(df.duplicated(subset=keys).sum())
            if dupes:
                issues.append(f"{dupes:,} filas duplicadas en la llave {keys}")

        for c in schema["required"]:
            if c in df.columns:
                null_pct = float(df[c].isna().mean())
                if null_pct > 0.02:
                    issues.append(f"'{c}' tiene {null_pct:.1%} de nulos")

        if issues:
            problems[name] = issues

    if problems:
        print("\n⚠️  Problemas de validación:")
        for t, iss in problems.items():
            for i in iss:
                print(f"   [{t}] {i}")
        if strict:
            raise ValueError("Validación fallida (strict=True)")
    else:
        print("✓ Validación de esquemas: sin problemas")
    return problems


def profile(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Perfil de calidad por tabla: filas, llaves distintas, rango de fechas,
    % de nulos promedio. Es el primer paso del EDA."""
    rows = []
    for name, df in tables.items():
        if df is None or df.empty:
            rows.append({"tabla": name, "filas": 0, "columnas": 0})
            continue
        schema = TABLE_SCHEMAS.get(name, {})
        rec = {
            "tabla": name,
            "filas": len(df),
            "columnas": df.shape[1],
            "nulos_prom_%": round(float(df.isna().mean().mean()) * 100, 2),
            "mem_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
        }
        for k in ("cif_id", "account_id", "loan_id"):
            if k in df.columns:
                rec[f"{k}_distintos"] = int(df[k].nunique())
        dcol = "as_of_month" if "as_of_month" in df.columns else (
            schema.get("dates", [None])[0])
        if dcol and dcol in df.columns and df[dcol].notna().any():
            rec["desde"] = str(df[dcol].min().date())
            rec["hasta"] = str(df[dcol].max().date())
        rows.append(rec)
    return pd.DataFrame(rows)


def _print_summary(tables: dict[str, pd.DataFrame]) -> None:
    total = sum(len(d) for d in tables.values() if d is not None)
    print(f"\n📦 {len([d for d in tables.values() if d is not None and not d.empty])}"
          f" tablas cargadas, {total:,} filas en total")


def save_processed(df: pd.DataFrame, name: str, cfg: Config) -> Path:
    """Guarda un artefacto intermedio (panel, features) en processed_dir."""
    path = cfg.processed_path() / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"💾 Guardado: {path}  ({len(df):,} filas)")
    return path


def load_processed(name: str, cfg: Config) -> pd.DataFrame | None:
    path = cfg.processed_path() / f"{name}.parquet"
    if path.exists():
        print(f"📂 Reusando: {path}")
        return pd.read_parquet(path)
    return None

## 5 · Panel, ownership y target

Donde se materializan las tres decisiones de diseño del modelo.

**Saldos ponderados por rol.** Una cuenta joint no cuenta entera para cada titular
(`1/n`), y un authorized signer no cuenta nada. Sin esto, los depósitos del banco
quedan inflados por doble conteo.

**Target compuesto.** No basta con el cierre formal: la attrition silenciosa —saldo
bajo **y** sin préstamos **y** sin actividad— sostenida durante
`target.persistence_months` cuenta igual. Es la que se escapa de los reportes.

**La espina es el maestro de clientes**, no las relaciones vigentes. Si se
construyera sobre relaciones, el cliente que cierra todo desaparecería del panel
justo en el mes que interesa, en vez de aparecer con ceros.

In [ ]:
# Roles que cuentan como préstamo (para exposición crediticia)
LOAN_ROLES = {"borrower", "co_borrower", "guarantor"}


# ── Pesos de ownership ──────────────────────────────────────────────────

def attach_ownership_weights(relationships: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Añade `ownership_weight` a cada fila (cif, account, role).

    Peso `null` en config = repartir 1/n_owners entre los titulares, que es lo
    que evita el doble conteo de saldos en cuentas joint.
    """
    rel = relationships.copy()
    weights = cfg.ownership_weights
    rel["relationship_role"] = rel["relationship_role"].astype(str).str.lower().str.strip()

    if "n_owners" not in rel.columns or rel["n_owners"].isna().all():
        # Derivar de los datos si el extracto no lo trae
        owner_roles = {r for r, w in weights.items() if w is None or w > 0}
        counts = (rel[rel["relationship_role"].isin(owner_roles)]
                  .groupby("account_id")["cif_id"].nunique().rename("n_owners"))
        rel = rel.drop(columns=[c for c in ["n_owners"] if c in rel.columns])
        rel = rel.merge(counts, on="account_id", how="left")
    rel["n_owners"] = rel["n_owners"].fillna(1).clip(lower=1)

    explicit = rel["relationship_role"].map(
        {k: v for k, v in weights.items() if v is not None}
    )
    shared = 1.0 / rel["n_owners"]
    # Rol conocido con peso explícito → ese peso.
    # Rol conocido con peso null → 1/n_owners.
    # Rol desconocido → 0 (conservador: no inventamos exposición).
    known_null = rel["relationship_role"].isin(
        [k for k, v in weights.items() if v is None]
    )
    rel["ownership_weight"] = np.where(
        explicit.notna(), explicit.fillna(0.0),
        np.where(known_null, shared, 0.0),
    )
    return rel


def _active_relationships(rel: pd.DataFrame, months: pd.DatetimeIndex) -> pd.DataFrame:
    """Expande relaciones a (cif, account, role, mes) para los meses vigentes."""
    grid = pd.DataFrame({"as_of_month": months, "_k": 1})
    rel = rel.assign(_k=1)
    out = rel.merge(grid, on="_k").drop(columns="_k")
    start_ok = out["relationship_start_date"].isna() | \
               (out["relationship_start_date"] <= out["as_of_month"])
    end_ok = out["relationship_end_date"].isna() | \
             (out["relationship_end_date"] > out["as_of_month"])
    return out[start_ok & end_ok].reset_index(drop=True)


# ── Estado mensual por CIF ──────────────────────────────────────────────

def build_cif_month_state(tables: dict[str, pd.DataFrame], cfg: Config) -> pd.DataFrame:
    """Agrega todo a una fila por (cif_id, as_of_month) con el estado de la
    relación: saldos ponderados, productos, préstamos, actividad."""
    deposits = tables["deposits"]
    months = pd.DatetimeIndex(sorted(deposits["as_of_month"].unique()))

    rel = attach_ownership_weights(tables["relationships"], cfg)
    rel_m = _active_relationships(rel, months)

    # ── Depósitos ponderados por ownership ──
    dep_cols = ["account_id", "as_of_month", "eom_balance", "product_family"]
    for c in ("nsf_count_month", "fees_charged_month", "account_status",
              "maturity_date", "interest_rate_paid", "close_reason_code"):
        if c in deposits.columns:
            dep_cols.append(c)
    dep = deposits[dep_cols]

    dm = rel_m[rel_m["product_type"] == "deposit"].merge(
        dep, on=["account_id", "as_of_month"], how="inner"
    )
    dm["owned_balance"] = dm["eom_balance"] * dm["ownership_weight"]
    dm["is_owner"] = dm["ownership_weight"] > 0

    agg = {
        "owned_balance": ("owned_balance", "sum"),
        "raw_balance_any_role": ("eom_balance", "sum"),
        "n_deposit_accounts": ("account_id", "nunique"),
    }
    if "nsf_count_month" in dm.columns:
        agg["nsf_count"] = ("nsf_count_month", "sum")
    if "fees_charged_month" in dm.columns:
        agg["fees_charged"] = ("fees_charged_month", "sum")

    dep_state = dm[dm["is_owner"]].groupby(["cif_id", "as_of_month"]).agg(**agg).reset_index()

    # Conteos por familia de producto (solo donde hay ownership)
    fam = (dm[dm["is_owner"]]
           .pivot_table(index=["cif_id", "as_of_month"], columns="product_family",
                        values="account_id", aggfunc="nunique", fill_value=0)
           .add_prefix("n_").reset_index())
    dep_state = dep_state.merge(fam, on=["cif_id", "as_of_month"], how="left")

    # Relacional: en cuántas cuentas aparece con CUALQUIER rol (incl. signer)
    rel_state = (rel_m.groupby(["cif_id", "as_of_month"])
                 .agg(n_relationships_any_role=("account_id", "nunique"),
                      n_signer_only=("relationship_role",
                                     lambda s: int((s == "authorized_signer").sum())),
                      n_joint_accounts=("relationship_role",
                                        lambda s: int((s == "joint").sum())))
                 .reset_index())

    # ── Espina del panel: el maestro de clientes, NO las relaciones vigentes ──
    # Si se parte de las relaciones, el cliente que cierra todos sus productos
    # simplemente desaparece del panel en vez de aparecer con ceros, y la
    # condición "sin relaciones" del target se vuelve inalcanzable. El maestro
    # de clientes sí conserva la fila mes a mes.
    party_all = tables["party"]
    spine = party_all[["cif_id", "as_of_month"]].drop_duplicates()
    spine = spine[spine["as_of_month"].isin(months)]

    state = (spine
             .merge(rel_state, on=["cif_id", "as_of_month"], how="left")
             .merge(dep_state, on=["cif_id", "as_of_month"], how="left"))

    # CD próximo a vencer: gatillo clásico de rolloff
    if "maturity_date" in dm.columns:
        cd = dm[(dm["is_owner"]) & (dm["product_family"].astype(str) == "cd")].copy()
        if len(cd):
            cd["months_to_cd_maturity"] = (
                (cd["maturity_date"].dt.to_period("M").astype("int64")
                 - cd["as_of_month"].dt.to_period("M").astype("int64"))
            )
            cdg = (cd.groupby(["cif_id", "as_of_month"])
                   .agg(months_to_cd_maturity=("months_to_cd_maturity", "min"),
                        cd_rate_avg=("interest_rate_paid", "mean")).reset_index())
            state = state.merge(cdg, on=["cif_id", "as_of_month"], how="left")

    # ── Préstamos ──
    state = _add_loan_state(state, tables, rel_m, cfg)

    # ── Actividad transaccional ──
    state = _add_activity_state(state, tables, rel_m)

    # ── Digital ──
    digital = tables.get("digital")
    if digital is not None and not digital.empty:
        dcols = [c for c in digital.columns if c not in ("cif_id", "as_of_month")]
        state = state.merge(digital[["cif_id", "as_of_month"] + dcols],
                            on=["cif_id", "as_of_month"], how="left")

    # ── Atributos del cliente ──
    party = tables["party"]
    pcols = [c for c in ["entity_type", "segment", "residency_status", "state",
                         "tenure_months", "employee_flag", "deceased_flag",
                         "kyc_risk_rating", "customer_since_date", "household_id"]
             if c in party.columns]
    state = state.merge(party[["cif_id", "as_of_month"] + pcols],
                        on=["cif_id", "as_of_month"], how="left")

    # Un conteo o un monto ausente sí es un 0. Un "meses hasta…" ausente no:
    # rellenarlo con 0 diría "vence este mes", que es exactamente lo contrario
    # de "este cliente no tiene ese producto". Se quedan en NaN para que las
    # features de dominio puedan distinguirlos y el imputador de los modelos
    # los marque como faltantes.
    NOT_APPLICABLE = {"min_months_to_payoff", "months_to_cd_maturity", "cd_rate_avg"}
    num = [c for c in state.select_dtypes(include=[np.number]).columns
           if c not in NOT_APPLICABLE]
    state[num] = state[num].fillna(0)
    return state.sort_values(["cif_id", "as_of_month"]).reset_index(drop=True)


def _add_loan_state(state, tables, rel_m, cfg) -> pd.DataFrame:
    loans = tables.get("loans")
    if loans is None or loans.empty:
        state["n_active_loans"] = 0
        state["loan_balance"] = 0.0
        state["min_months_to_payoff"] = np.nan
        return state

    lm = rel_m[rel_m["relationship_role"].isin(LOAN_ROLES)].merge(
        loans.rename(columns={"loan_id": "account_id"}),
        on=["account_id", "as_of_month"], how="inner",
    )
    if lm.empty:
        state["n_active_loans"] = 0
        state["loan_balance"] = 0.0
        state["min_months_to_payoff"] = np.nan
        return state

    lm["is_active"] = (lm.get("status", pd.Series("active", index=lm.index))
                       .astype(str).eq("active")) & (lm["current_balance"] > 0)
    lm["owned_loan_balance"] = lm["current_balance"] * lm["ownership_weight"].clip(lower=0)

    # Meses hasta liquidar según amortización: el gatillo de fuga que
    # describe el README (loan por terminar + sin depósitos activos).
    if "months_to_maturity" in lm.columns:
        lm["months_to_payoff"] = lm["months_to_maturity"]
    elif "remaining_term_months" in lm.columns:
        lm["months_to_payoff"] = lm["remaining_term_months"]
    else:
        lm["months_to_payoff"] = np.nan

    agg = (lm.groupby(["cif_id", "as_of_month"]).agg(
        n_active_loans=("is_active", "sum"),
        loan_balance=("owned_loan_balance", "sum"),
        min_months_to_payoff=("months_to_payoff", "min"),
        max_days_past_due=("days_past_due", "max") if "days_past_due" in lm.columns
                          else ("is_active", "size"),
    ).reset_index())

    return state.merge(agg, on=["cif_id", "as_of_month"], how="left")


def _add_activity_state(state, tables, rel_m) -> pd.DataFrame:
    txns = tables.get("transactions")
    if txns is None or txns.empty:
        state["txn_count_total"] = np.nan
        state["months_since_activity"] = np.nan
        state["has_direct_deposit"] = np.nan
        state["competitor_outflow_amt"] = np.nan
        return state

    owners = rel_m[rel_m["ownership_weight"] > 0][
        ["cif_id", "account_id", "as_of_month"]
    ].drop_duplicates()
    tx = owners.merge(txns, on=["account_id", "as_of_month"], how="inner")

    aggs = {"txn_count_total": ("txn_count", "sum")}
    if "total_amount_debit" in tx.columns:
        aggs["debit_amt_total"] = ("total_amount_debit", "sum")
    if "total_amount_credit" in tx.columns:
        aggs["credit_amt_total"] = ("total_amount_credit", "sum")
    g = tx.groupby(["cif_id", "as_of_month"]).agg(**aggs).reset_index()

    # Nómina y fuga a la competencia. En formato largo llegan como banderas por
    # fila; en formato ancho eso no cabe —la fila es la cuenta, no el par
    # categoría×mes—, así que el extracto los trae ya agregados como métricas
    # mensuales propias. Se aceptan las dos formas.
    if "is_direct_deposit" in tx.columns:
        dd = (tx[tx["is_direct_deposit"].astype("boolean").fillna(False)]
              .groupby(["cif_id", "as_of_month"])["txn_count"].sum()
              .rename("dd_txn_count").reset_index())
        g = g.merge(dd, on=["cif_id", "as_of_month"], how="left")
    elif "dd_txn_count" in tx.columns:
        # Viene por cuenta y mes, repetido en cada fila de categoría. Hay que
        # deduplicar antes de sumar entre cuentas: sumar tal cual lo
        # multiplicaría por el número de categorías, y quedarse con el máximo
        # perdería la segunda cuenta con nómina del mismo cliente.
        dd = (tx.drop_duplicates(["cif_id", "account_id", "as_of_month"])
              .groupby(["cif_id", "as_of_month"])["dd_txn_count"].sum()
              .reset_index())
        g = g.merge(dd, on=["cif_id", "as_of_month"], how="left")
    if "dd_txn_count" in g.columns:
        g["dd_txn_count"] = g["dd_txn_count"].fillna(0)
        g["has_direct_deposit"] = (g["dd_txn_count"] > 0).astype(int)

    if "competitor_transfer_flag" in tx.columns:
        cp = (tx[tx["competitor_transfer_flag"].astype("boolean").fillna(False)]
              .groupby(["cif_id", "as_of_month"])
              .agg(competitor_outflow_amt=("total_amount_debit", "sum"),
                   competitor_txn_count=("txn_count", "sum")).reset_index())
        g = g.merge(cp, on=["cif_id", "as_of_month"], how="left")
    elif "competitor_outflow_amt" in tx.columns:
        agg_cp = {"competitor_outflow_amt": ("competitor_outflow_amt", "sum")}
        if "competitor_txn_count" in tx.columns:
            agg_cp["competitor_txn_count"] = ("competitor_txn_count", "sum")
        cp = (tx.drop_duplicates(["cif_id", "account_id", "as_of_month"])
              .groupby(["cif_id", "as_of_month"]).agg(**agg_cp).reset_index())
        g = g.merge(cp, on=["cif_id", "as_of_month"], how="left")
    for c in ("competitor_outflow_amt", "competitor_txn_count"):
        if c in g.columns:
            g[c] = g[c].fillna(0)

    # Categorías principales en columnas separadas
    top = ["ach_credit_in", "ach_debit_out", "zelle_out", "pos_signature",
           "bill_pay", "atm_foreign", "wire_out", "branch_deposit", "mobile_deposit"]
    piv = (tx[tx["txn_category"].isin(top)]
           .pivot_table(index=["cif_id", "as_of_month"], columns="txn_category",
                        values="txn_count", aggfunc="sum", fill_value=0)
           .add_prefix("txn_").reset_index())
    g = g.merge(piv, on=["cif_id", "as_of_month"], how="left")

    state = state.merge(g, on=["cif_id", "as_of_month"], how="left")
    state["txn_count_total"] = state["txn_count_total"].fillna(0)

    # Meses consecutivos sin actividad iniciada por el cliente
    state = state.sort_values(["cif_id", "as_of_month"])
    active = state["txn_count_total"] > 0
    grp = active.groupby(state["cif_id"]).cumsum()
    state["months_since_activity"] = (
        state.groupby(["cif_id", grp]).cumcount().where(~active, 0)
    )
    return state


# ── Target ──────────────────────────────────────────────────────────────

def label_attrition_state(state: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Marca, mes a mes, si el CIF está en estado de attrition, y deriva el
    mes del evento (primer mes de una racha sostenida)."""
    t = cfg.target
    df = state.sort_values(["cif_id", "as_of_month"]).copy()

    # Baseline propio: promedio de saldo de los 12 meses previos (shift para
    # no usar el mes corriente).
    df["balance_baseline_12m"] = (
        df.groupby("cif_id")["owned_balance"]
          .transform(lambda s: s.shift(1).rolling(12, min_periods=3).mean())
    )
    thresh = np.maximum(
        float(t.get("low_balance_abs", 100.0)),
        df["balance_baseline_12m"].fillna(0) * float(t.get("low_balance_pct_of_baseline", 0.05)),
    )
    low_balance = df["owned_balance"] < thresh
    no_loans = df.get("n_active_loans", pd.Series(0, index=df.index)).fillna(0) <= 0

    inactivity_months = max(1, int(round(float(t.get("inactivity_days", 90)) / 30)))
    if df["months_since_activity"].notna().any():
        inactive = df["months_since_activity"].fillna(0) >= inactivity_months
    else:
        inactive = pd.Series(True, index=df.index)  # sin datos de txn: no restringe

    no_relationship = df["n_relationships_any_role"].fillna(0) <= 0

    df["is_attrited_state"] = (low_balance & no_loans & inactive) | no_relationship

    # Confirmar con persistencia de N meses consecutivos
    k = int(t.get("persistence_months", 2))
    if k > 1:
        rolled = (df.groupby("cif_id")["is_attrited_state"]
                  .transform(lambda s: s.rolling(k, min_periods=k).sum()))
        confirmed = rolled >= k
        # El evento se fecha en el PRIMER mes de la racha, no en el último
        df["_confirmed"] = confirmed
        df["churn_flag"] = (df.groupby("cif_id")["_confirmed"]
                            .transform(lambda s: s.shift(-(k - 1)).fillna(False)))
        df = df.drop(columns="_confirmed")
    else:
        df["churn_flag"] = df["is_attrited_state"]

    # Mes del evento = primer mes con churn_flag
    ev = (df[df["churn_flag"].astype(bool)]
          .groupby("cif_id")["as_of_month"].min().rename("churn_event_month"))
    df = df.merge(ev, on="cif_id", how="left")
    return df


def _involuntary_months(tables: dict[str, pd.DataFrame], cfg: Config) -> pd.Series:
    """Primer mes con causa de salida involuntaria por CIF (para excluir)."""
    codes = {c.upper() for c in cfg.target.get("involuntary_reason_codes", [])}
    frames = []

    events = tables.get("events")
    if events is not None and not events.empty and "reason_code" in events.columns:
        e = events[events["reason_code"].astype(str).str.upper().isin(codes)]
        if len(e):
            frames.append(e.groupby("cif_id")["event_date"].min())

    party = tables.get("party")
    # `deceased_date` manda sobre `deceased_flag`: la bandera solo dice "hoy
    # está fallecido", no desde cuándo. En un maestro de clientes sin
    # dimensión temporal es lo único utilizable — con la bandera sola, el
    # cliente queda marcado durante toda su historia y se excluye del panel
    # entero, en vez de desde el evento.
    if party is not None and "deceased_date" in party.columns:
        d = party.assign(_d=pd.to_datetime(party["deceased_date"], errors="coerce"))
        d = d.dropna(subset=["_d"])
        if len(d):
            frames.append(d.groupby("cif_id")["_d"].min())
    elif party is not None and "deceased_flag" in party.columns \
            and "as_of_month" in party.columns:
        d = party[party["deceased_flag"].astype("boolean").fillna(False)]
        if len(d):
            frames.append(d.groupby("cif_id")["as_of_month"].min())

    if not frames:
        return pd.Series(dtype="datetime64[ns]", name="involuntary_month")
    s = pd.concat(frames).groupby(level=0).min()
    s.name = "involuntary_month"
    return s


def build_panel(tables: dict[str, pd.DataFrame], cfg: Config,
                verbose: bool = True,
                state: pd.DataFrame | None = None) -> pd.DataFrame:
    """Panel final: una fila por (cif_id, as_of_month) elegible, con etiqueta `y`.

    `y = 1` si el CIF tiene su evento de churn dentro de la ventana de
    performance, que empieza `gap_months + 1` meses después del as-of date.

    `state` permite reusar un estado ya calculado (con o sin etiquetar) en vez
    de reconstruirlo: el notebook lo arma antes para inspeccionarlo, y volver
    a calcularlo aquí duplicaría la parte más lenta del pipeline.
    """
    if state is None:
        state = build_cif_month_state(tables, cfg)
    if "is_attrited_state" not in state.columns:
        state = label_attrition_state(state, cfg)

    p = cfg.panel
    gap = int(p.get("gap_months", 1))
    perf = int(p.get("performance_window_months", 6))
    feat_win = int(p.get("feature_window_months", 12))
    min_tenure = int(p.get("min_tenure_months", 6))
    # Historia mínima real que exigimos por detrás. Es deliberadamente menor
    # que feature_window: los rolling usan min_periods, así que una ventana
    # parcial sigue produciendo features útiles. Exigir los 12 meses completos
    # tira un año entero de panel sin necesidad.
    min_hist = int(p.get("min_history_months", max(3, feat_win // 2)))

    months = pd.DatetimeIndex(sorted(state["as_of_month"].unique()))
    # No se puede etiquetar sin ventana de performance completa por delante.
    first_valid = months[min(min_hist, len(months) - 1)]
    last_valid_pos = len(months) - (gap + perf) - 1
    if last_valid_pos < min_hist:
        raise ValueError(
            f"Historia insuficiente: {len(months)} meses no alcanzan para "
            f"min_history={min_hist} + gap={gap} + performance={perf}.\n"
            f"  → Necesitas al menos {min_hist + gap + perf + 2} meses, o "
            f"reduce performance_window_months / min_history_months."
        )
    last_valid = months[last_valid_pos]

    panel = state[(state["as_of_month"] >= first_valid) &
                  (state["as_of_month"] <= last_valid)].copy()

    # ── Etiqueta ──
    def _add_months(s: pd.Series, k: int) -> pd.Series:
        return (s.dt.to_period("M") + k).dt.to_timestamp()

    panel["label_start"] = _add_months(panel["as_of_month"], gap + 1)
    panel["label_end"] = _add_months(panel["as_of_month"], gap + perf)
    ev = panel["churn_event_month"]
    panel["y"] = ((ev.notna()) & (ev >= panel["label_start"]) &
                  (ev <= panel["label_end"])).astype(int)

    # ── Elegibilidad ──
    n0 = len(panel)
    # 1. No se puede predecir la fuga de quien ya se fue
    already_gone = ev.notna() & (panel["as_of_month"] >= ev)
    panel = panel[~already_gone]

    # 2. Antigüedad mínima
    if "tenure_months" in panel.columns:
        panel = panel[panel["tenure_months"].fillna(0) >= min_tenure]

    # 3. Debe tener al menos un rol elegible vigente
    panel = panel[panel["n_relationships_any_role"].fillna(0) > 0]

    # 4. Empleados fuera
    if "employee_flag" in panel.columns:
        panel = panel[~panel["employee_flag"].astype("boolean").fillna(False)]

    # 5. Salidas involuntarias: fuera el cliente desde antes del evento, para
    #    que el modelo no aprenda a predecir compliance/fallecimiento
    if cfg.target.get("exclude_involuntary", True):
        inv = _involuntary_months(tables, cfg)
        if len(inv):
            panel = panel.merge(inv, on="cif_id", how="left")
            bad = panel["involuntary_month"].notna() & \
                  (panel["label_end"] >= panel["involuntary_month"])
            panel = panel[~bad].drop(columns="involuntary_month")

    if verbose:
        rate = panel["y"].mean() if len(panel) else 0
        print(f"\n📊 Panel: {len(panel):,} filas  "
              f"({panel['cif_id'].nunique():,} CIFs × "
              f"{panel['as_of_month'].nunique()} meses)")
        print(f"   Descartadas por elegibilidad: {n0 - len(panel):,}")
        print(f"   Tasa de churn: {rate:.2%}  ({int(panel['y'].sum()):,} positivos)")
        print(f"   Ventana: as-of + gap {gap}m → performance {perf}m")

    return panel.reset_index(drop=True)

## 6 · Feature engineering

Para cada fila `(cif_id, as_of_month)`, **todo** se calcula sobre la ventana hacia
atrás `[t-11, t]`. Nunca sobre el futuro.

El churn es un proceso, no un atributo: un saldo de $5.000 estable y uno de $5.000
que venía de $50.000 son el mismo nivel y riesgos opuestos. Nueve familias de señal:

| Familia | Qué captura |
|:---|:---|
| Nivel | El valor de hoy — saldo, productos, logins |
| Tendencia | Pendiente OLS sobre la ventana |
| Ratio a sí mismo | El cliente contra su propia línea base |
| Drawdown | Caída desde el máximo de la ventana |
| Volatilidad | Dispersión de la serie |
| Aceleración | Si la caída se profundiza o se estabiliza |
| Rachas | Meses consecutivos en una misma condición |
| Estacionalidad | Patrones recurrentes por mes del año |
| Relacionales | Contagio — el co-titular que ya se está yendo |

In [ ]:
# Columnas que reciben el tratamiento temporal completo
TREND_COLS = [
    "owned_balance", "txn_count_total", "n_deposit_accounts",
    "n_relationships_any_role", "loan_balance", "login_count_mobile_month",
    "login_count_web_month", "bill_pay_payees_active", "debit_amt_total",
    "txn_pos_signature", "txn_ach_credit_in", "txn_zelle_out", "txn_bill_pay",
]

# Columnas donde interesa la suma acumulada en ventana (eventos, no niveles)
SUM_COLS = [
    "nsf_count", "fees_charged", "complaints_open_month", "disputes_month",
    "call_center_calls_month", "branch_visits_month", "competitor_outflow_amt",
    "competitor_txn_count", "txn_atm_foreign", "txn_wire_out",
]

# Se arrastran tal cual (estado en t)
LEVEL_COLS = [
    "n_active_loans", "min_months_to_payoff", "months_to_cd_maturity",
    "cd_rate_avg", "has_direct_deposit", "months_since_activity",
    "n_signer_only", "n_joint_accounts", "max_days_past_due",
    "n_checking", "n_savings", "n_mma", "n_cd",
    "online_enrolled_flag", "mobile_enrolled_flag", "estatement_flag",
    "debit_card_active_flag", "alerts_enrolled_flag", "tenure_months",
]

CATEGORICAL_COLS = ["entity_type", "segment", "residency_status", "state",
                    "kyc_risk_rating"]


# ── Utilidades temporales ───────────────────────────────────────────────

def _slope_weights(window: int) -> np.ndarray:
    """Pesos de la pendiente OLS para x = 0..w-1, ordenados de más antiguo a
    más reciente. slope = Σ wᵢ·y_{t-(w-1-i)}"""
    x = np.arange(window, dtype=float)
    xc = x - x.mean()
    return xc / (xc**2).sum()


def _rolling_slope(df: pd.DataFrame, col: str, window: int, group: str) -> pd.Series:
    """Pendiente OLS de `col` sobre los últimos `window` meses, por grupo.

    Vectorizado: una combinación lineal de `window` shifts, en vez de
    rolling().apply(), que sobre paneles grandes es órdenes de magnitud
    más lento.
    """
    w = _slope_weights(window)
    g = df.groupby(group)[col]
    out = None
    for i in range(window):
        lag = window - 1 - i
        s = g.shift(lag) * w[i]
        out = s if out is None else out + s
    return out


def _safe_ratio(num: pd.Series, den: pd.Series, cap: float = 50.0) -> pd.Series:
    """Ratio robusto: evita división por cero y trunca colas absurdas."""
    d = den.replace(0, np.nan)
    r = num / d
    return r.clip(-cap, cap)


# ── Motor principal ─────────────────────────────────────────────────────

def build_features(panel: pd.DataFrame, cfg: Config,
                   tables: dict[str, pd.DataFrame] | None = None,
                   verbose: bool = True) -> pd.DataFrame:
    """Devuelve el panel con todas las features temporales añadidas.

    Preserva `cif_id`, `as_of_month`, `y` para el split posterior.
    """
    df = panel.sort_values(["cif_id", "as_of_month"]).copy()
    g = "cif_id"
    win = int(cfg.panel.get("feature_window_months", 12))
    present_trend = [c for c in TREND_COLS if c in df.columns]
    new: dict[str, pd.Series] = {}

    # ── 1-6. Tratamiento temporal por columna de tendencia ──
    for col in present_trend:
        s = df[col].astype(float)
        grp = df.groupby(g)[col]

        # Ratios a su propio pasado
        for lag in (1, 3, 6, 12):
            if lag > win:
                continue
            new[f"{col}_ratio_{lag}m"] = _safe_ratio(s, grp.shift(lag))
            new[f"{col}_diff_{lag}m"] = s - grp.shift(lag)

        # Pendientes
        for w in (3, 6, 12):
            if w > win:
                continue
            new[f"{col}_slope_{w}m"] = _rolling_slope(df, col, w, g)

        # Aceleración: ¿la caída se está profundizando o estabilizando?
        if 6 <= win:
            slope_recent = new.get(f"{col}_slope_3m")
            slope_prev = df.groupby(g)[col].shift(3)
            if slope_recent is not None:
                prev_slope = _rolling_slope(
                    df.assign(_lagged=slope_prev), "_lagged", 3, g
                )
                new[f"{col}_accel"] = slope_recent - prev_slope

        # Estadísticos de ventana (causales: incluyen t)
        roll = grp.rolling(win, min_periods=2)
        mean_w = roll.mean().reset_index(level=0, drop=True)
        std_w = roll.std().reset_index(level=0, drop=True)
        max_w = roll.max().reset_index(level=0, drop=True)
        min_w = roll.min().reset_index(level=0, drop=True)

        new[f"{col}_mean_{win}m"] = mean_w
        new[f"{col}_cv_{win}m"] = _safe_ratio(std_w, mean_w.abs())
        # Drawdown: qué fracción de su propio máximo conserva.
        # Es de las features más predictivas y casi nunca se incluye.
        new[f"{col}_drawdown_{win}m"] = _safe_ratio(s, max_w, cap=5.0)
        new[f"{col}_vs_min_{win}m"] = _safe_ratio(s, min_w, cap=50.0)
        new[f"{col}_zscore_{win}m"] = _safe_ratio(s - mean_w, std_w, cap=10.0)

        # 7. Racha: meses consecutivos de caída
        declining = (s < grp.shift(1)).fillna(False)
        blocks = (~declining).groupby(df[g]).cumsum()
        new[f"{col}_declining_streak"] = (
            declining.groupby([df[g], blocks]).cumsum()
        )

    # ── Sumas en ventana para columnas de evento ──
    for col in [c for c in SUM_COLS if c in df.columns]:
        grp = df.groupby(g)[col]
        for w in (3, 6, 12):
            if w > win:
                continue
            new[f"{col}_sum_{w}m"] = (
                grp.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
            )

    out = pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

    # ── 7b. Rachas específicas de dominio ──
    out = _add_streak_features(out, g)

    # ── 8. Estacionalidad ──
    out["cal_month"] = out["as_of_month"].dt.month
    out["cal_quarter"] = out["as_of_month"].dt.quarter

    # ── 9. Features de dominio y relacionales ──
    out = _add_domain_features(out)
    if tables is not None:
        out = _add_contagion_features(out, tables, cfg)

    if verbose:
        n_feat = out.shape[1] - df.shape[1]
        print(f"🔧 Features generadas: {n_feat} nuevas "
              f"({out.shape[1]} columnas totales)")
    return out


def _add_streak_features(df: pd.DataFrame, g: str) -> pd.DataFrame:
    """Rachas y tiempos-desde-evento que no salen del tratamiento genérico."""
    if "has_direct_deposit" in df.columns:
        dd = df["has_direct_deposit"].fillna(0).astype(int)
        # Meses desde que perdió la nómina: el predictor #1 en checking
        had_dd = dd.groupby(df[g]).cummax()
        lost = (had_dd == 1) & (dd == 0)
        blocks = (~lost).groupby(df[g]).cumsum()
        df["months_since_dd_loss"] = lost.groupby([df[g], blocks]).cumsum()
        df["ever_had_direct_deposit"] = had_dd
        # ¿Perdió la nómina en los últimos 6 meses?
        df["dd_lost_recently"] = (
            (df["months_since_dd_loss"] > 0) & (df["months_since_dd_loss"] <= 6)
        ).astype(int)

    if "months_since_activity" in df.columns:
        df["is_dormant_3m"] = (df["months_since_activity"].fillna(0) >= 3).astype(int)

    return df


def _add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ratios y banderas específicas de banca."""
    # Loan por terminar sin ancla de depósitos: el escenario del README.
    # La condición de tener préstamo vivo es explícita: sin ella, cualquier
    # cliente sin préstamos y con poco saldo entraría en la bandera, que es
    # justo la población que la feature pretende excluir.
    if {"min_months_to_payoff", "owned_balance"} <= set(df.columns):
        near_payoff = df["min_months_to_payoff"].fillna(999) <= 6
        if "n_active_loans" in df.columns:
            near_payoff &= df["n_active_loans"].fillna(0) > 0
        thin_deposits = df["owned_balance"].fillna(0) < 500
        df["loan_ending_no_deposit_anchor"] = (near_payoff & thin_deposits).astype(int)
        df["months_to_payoff_capped"] = df["min_months_to_payoff"].fillna(999).clip(0, 120)

    # CD por vencer: gatillo de rolloff. Mismo cuidado — solo aplica a quien
    # efectivamente tiene un CD.
    if "months_to_cd_maturity" in df.columns:
        cd_maturing = df["months_to_cd_maturity"].fillna(999).between(0, 3)
        if "n_cd" in df.columns:
            cd_maturing &= df["n_cd"].fillna(0) > 0
        df["cd_maturing_3m"] = cd_maturing.astype(int)

    # Profundidad de relación
    prod_cols = [c for c in ("n_checking", "n_savings", "n_mma", "n_cd") if c in df.columns]
    if prod_cols:
        df["n_products_owned"] = df[prod_cols].fillna(0).sum(axis=1)
        df["product_diversity"] = (df[prod_cols].fillna(0) > 0).sum(axis=1)
    if "n_active_loans" in df.columns and "n_products_owned" in df.columns:
        df["has_deposit_and_loan"] = (
            (df["n_products_owned"] > 0) & (df["n_active_loans"] > 0)
        ).astype(int)

    # Intensidad de uso normalizada por saldo
    if {"txn_count_total", "owned_balance"} <= set(df.columns):
        df["txn_per_1k_balance"] = _safe_ratio(
            df["txn_count_total"], df["owned_balance"] / 1000
        )

    # Fuga de fondos hacia otra institución
    if {"competitor_outflow_amt_sum_6m", "owned_balance"} <= set(df.columns):
        df["competitor_outflow_ratio"] = _safe_ratio(
            df["competitor_outflow_amt_sum_6m"], df["owned_balance"], cap=20.0
        )

    # Presión de fees respecto al saldo
    if {"fees_charged_sum_6m", "owned_balance"} <= set(df.columns):
        df["fee_burden_ratio"] = _safe_ratio(
            df["fees_charged_sum_6m"], df["owned_balance"], cap=5.0
        )

    # Digital engagement combinado
    logins = [c for c in ("login_count_web_month", "login_count_mobile_month")
              if c in df.columns]
    if logins:
        df["logins_total"] = df[logins].fillna(0).sum(axis=1)
        df["is_digitally_inactive"] = (df["logins_total"] == 0).astype(int)

    return df


def _add_contagion_features(df: pd.DataFrame, tables: dict[str, pd.DataFrame],
                            cfg: Config) -> pd.DataFrame:
    """Contagio de hogar: si el co-titular de una cuenta joint se va, el
    riesgo del CIF sube muchísimo. Suele ser la feature relacional más fuerte.
    """
    rel = tables.get("relationships")
    if rel is None or rel.empty or "churn_event_month" not in df.columns:
        return df

    joint = rel[rel["relationship_role"].astype(str).str.lower() == "joint"]
    if joint.empty:
        df["n_co_owners"] = 0
        df["co_owner_churned_flag"] = 0
        return df

    # Pares de co-titulares que comparten cuenta
    pairs = joint[["account_id", "cif_id"]].merge(
        joint[["account_id", "cif_id"]], on="account_id", suffixes=("", "_co")
    )
    pairs = pairs[pairs["cif_id"] != pairs["cif_id_co"]][["cif_id", "cif_id_co"]]
    pairs = pairs.drop_duplicates()

    n_co = pairs.groupby("cif_id")["cif_id_co"].nunique().rename("n_co_owners")

    # Mes de churn del co-titular (una fila por CIF)
    events = (df[["cif_id", "churn_event_month"]].dropna()
              .drop_duplicates("cif_id")
              .rename(columns={"cif_id": "cif_id_co",
                               "churn_event_month": "co_churn_month"}))
    pe = pairs.merge(events, on="cif_id_co", how="inner")
    if pe.empty:
        df["n_co_owners"] = df["cif_id"].map(n_co).fillna(0)
        df["co_owner_churned_flag"] = 0
        return df

    earliest = pe.groupby("cif_id")["co_churn_month"].min().rename("co_churn_month")
    df = df.merge(earliest, on="cif_id", how="left")
    df["n_co_owners"] = df["cif_id"].map(n_co).fillna(0)
    # Solo cuenta si el co-titular ya se fue AL MOMENTO del as-of date:
    # usar un churn futuro sería leakage.
    df["co_owner_churned_flag"] = (
        df["co_churn_month"].notna() & (df["co_churn_month"] <= df["as_of_month"])
    ).astype(int)
    df = df.drop(columns=["co_churn_month"])
    return df


# ── Selección de columnas para el modelo ────────────────────────────────

# Columnas que NUNCA deben entrar como feature: son el target, sus insumos
# directos, o identificadores.
LEAKY_COLS = {
    "y", "churn_flag", "is_attrited_state", "churn_event_month",
    "label_start", "label_end", "cif_id", "as_of_month",
    "balance_baseline_12m", "involuntary_month", "deceased_flag",
    "raw_balance_any_role", "customer_since_date",
}


def select_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    """Devuelve (features_numéricas, features_categóricas), excluyendo
    cualquier columna con riesgo de leakage."""
    numeric, categorical = [], []
    for c in df.columns:
        if c in LEAKY_COLS:
            continue
        if c in CATEGORICAL_COLS or df[c].dtype == object:
            if df[c].nunique(dropna=True) <= 50:
                categorical.append(c)
            continue
        if pd.api.types.is_bool_dtype(df[c]):
            numeric.append(c)
        elif pd.api.types.is_numeric_dtype(df[c]):
            numeric.append(c)
    return numeric, categorical


def clean_features(df: pd.DataFrame, numeric: list[str]) -> pd.DataFrame:
    """Reemplaza inf y recorta outliers extremos por winsorización suave."""
    out = df.copy()
    out[numeric] = out[numeric].replace([np.inf, -np.inf], np.nan)
    return out

## 7 · Split, métricas y calibración

Tres cosas distintas al default de sklearn:

1. **Split out-of-time con embargo.** No basta con cortar por fecha: una fila de
   train con as-of en noviembre y ventana de 6 meses "ve" hasta mayo. El embargo
   elimina del train toda fila cuya `label_end` caiga dentro del periodo de test.
2. **`StratifiedGroupKFold` agrupado por CIF.** Con panel, el mismo cliente aparece
   en muchas filas; un KFold normal lo pone en train y validación a la vez.
3. **Métricas de negocio.** Con 3-5% de tasa base, accuracy es inútil y ROC-AUC
   engaña. Lo que decide es lift y capture en los primeros deciles, porque retención
   solo puede contactar el top 5-10% de la lista.

In [ ]:
# ── Splits ──────────────────────────────────────────────────────────────

def temporal_split(df: pd.DataFrame, cfg: Config, verbose: bool = True):
    """Split out-of-time con embargo. Devuelve (train, test).

    El embargo cuesta `gap + performance` meses de panel: una fila de train con
    as-of en el mes t etiqueta hasta t+gap+perf, así que para no contaminar el
    periodo de test hace falta ese colchón. Es un costo real, no un bug — si
    deja el train vacío, el problema es que el panel es demasiado corto.
    """
    oot_months = int(cfg.split.get("oot_months", 6))
    use_embargo = bool(cfg.split.get("embargo", True))
    months = pd.DatetimeIndex(sorted(df["as_of_month"].unique()))
    if len(months) <= oot_months:
        raise ValueError(
            f"Solo hay {len(months)} meses en el panel; no alcanzan para "
            f"reservar {oot_months} como out-of-time."
        )
    cutoff = months[-oot_months]

    test = df[df["as_of_month"] >= cutoff].copy()
    train_raw = df[df["as_of_month"] < cutoff]

    # Embargo: fuera las filas de train que etiquetan dentro del periodo de test
    if use_embargo and "label_end" in train_raw.columns:
        train = train_raw[train_raw["label_end"] < cutoff].copy()
        embargoed = len(train_raw) - len(train)
    else:
        train, embargoed = train_raw.copy(), 0

    if len(train) == 0:
        gap = int(cfg.panel.get("gap_months", 1))
        perf = int(cfg.panel.get("performance_window_months", 6))
        raise ValueError(
            f"El embargo dejó el train vacío.\n"
            f"  Panel: {len(months)} meses ({months[0].date()} → {months[-1].date()})\n"
            f"  OOT reservado: {oot_months} meses (corte {cutoff.date()})\n"
            f"  Colchón que exige el embargo: gap({gap}) + performance({perf}) "
            f"= {gap + perf} meses\n"
            f"  → Necesitas un panel de al menos "
            f"{oot_months + gap + perf + 3} meses.\n"
            f"  Opciones: más historia, menos oot_months, "
            f"performance_window más corta, o split.embargo=false "
            f"(acepta un sesgo optimista leve)."
        )

    if verbose:
        print(f"\n✂️  Split out-of-time (corte: {cutoff.date()})")
        print(f"   Train: {len(train):>8,} filas  "
              f"({train['as_of_month'].min().date()} → "
              f"{train['as_of_month'].max().date()})  "
              f"churn {train['y'].mean():.2%}")
        print(f"   Test:  {len(test):>8,} filas  "
              f"({test['as_of_month'].min().date()} → "
              f"{test['as_of_month'].max().date()})  "
              f"churn {test['y'].mean():.2%}")
        print(f"   Embargadas por solapamiento de ventana: {embargoed:,}"
              f"{'' if use_embargo else '  (embargo DESACTIVADO)'}")
        overlap = len(set(train['cif_id']) & set(test['cif_id']))
        print(f"   CIFs presentes en ambos: {overlap:,} "
              f"(esperado y correcto en OOT: en producción re-scoreas al mismo cliente)")
    return train, test


def make_cv(cfg: Config) -> StratifiedGroupKFold:
    """CV estratificado Y agrupado por CIF. El `groups` va en `.split()`."""
    return StratifiedGroupKFold(
        n_splits=int(cfg.split.get("n_folds", 5)),
        shuffle=True,
        random_state=int(cfg.split.get("random_state", 42)),
    )


# ── Métricas ────────────────────────────────────────────────────────────

def lift_at_k(y_true, y_prob, k: float = 0.10) -> float:
    """Lift en el top k de la lista ordenada por probabilidad."""
    y_true = np.asarray(y_true)
    n = max(1, int(len(y_true) * k))
    idx = np.argsort(y_prob)[::-1][:n]
    base = y_true.mean()
    return float(y_true[idx].mean() / base) if base > 0 else 0.0


def capture_at_k(y_true, y_prob, k: float = 0.10) -> float:
    """Fracción de todos los churners capturada en el top k."""
    y_true = np.asarray(y_true)
    n = max(1, int(len(y_true) * k))
    idx = np.argsort(y_prob)[::-1][:n]
    total = y_true.sum()
    return float(y_true[idx].sum() / total) if total > 0 else 0.0


def evaluate(y_true, y_prob, prefix: str = "") -> dict[str, float]:
    """Panel de métricas. PR-AUC es la principal; lift/capture son el KPI
    que le importa al negocio."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob, dtype=float)
    m = {
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "log_loss": float(log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6))),
        "base_rate": float(y_true.mean()),
    }
    for k in (0.05, 0.10, 0.20):
        m[f"lift@{int(k*100)}"] = lift_at_k(y_true, y_prob, k)
        m[f"capture@{int(k*100)}"] = capture_at_k(y_true, y_prob, k)
    return {f"{prefix}{k}": v for k, v in m.items()}


def decile_table(y_true, y_prob, n_bins: int = 10) -> pd.DataFrame:
    """Tabla de deciles: la forma en que retención va a leer el modelo."""
    df = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(y_prob)})
    df = df.sort_values("p", ascending=False).reset_index(drop=True)
    df["decil"] = (np.arange(len(df)) * n_bins // len(df)) + 1

    base = df["y"].mean()
    out = (df.groupby("decil")
           .agg(n=("y", "size"), churners=("y", "sum"),
                tasa_churn=("y", "mean"), prob_media=("p", "mean"),
                prob_min=("p", "min"))
           .reset_index())
    out["lift"] = out["tasa_churn"] / base if base > 0 else 0
    out["capture_%"] = 100 * out["churners"].cumsum() / max(df["y"].sum(), 1)
    for c in ("tasa_churn", "prob_media", "prob_min", "lift", "capture_%"):
        out[c] = out[c].round(4)
    return out


def calibration_table(y_true, y_prob, n_bins: int = 10) -> pd.DataFrame:
    """Probabilidad predicha vs. observada. Si el modelo alimenta un cálculo
    de valor esperado, esta tabla tiene que estar cerca de la diagonal."""
    df = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(y_prob)})
    try:
        df["bin"] = pd.qcut(df["p"], n_bins, duplicates="drop", labels=False)
    except ValueError:
        df["bin"] = 0
    out = (df.groupby("bin")
           .agg(n=("y", "size"), prob_predicha=("p", "mean"),
                tasa_observada=("y", "mean")).reset_index())
    out["gap"] = (out["prob_predicha"] - out["tasa_observada"]).round(4)
    return out.round(4)


def calibrate(estimator, X_cal, y_cal, method: str = "isotonic"):
    """Calibración post-hoc sobre datos no usados en entrenamiento.

    El estimador ya está entrenado; aquí solo se ajusta el mapeo de
    probabilidad. En sklearn ≥1.6 eso se expresa con `FrozenEstimator`
    (`cv="prefit"` quedó deprecado y desaparece en 1.8).
    """
    try:
        from sklearn.frozen import FrozenEstimator

        cal = CalibratedClassifierCV(FrozenEstimator(estimator), method=method)
    except ImportError:
        cal = CalibratedClassifierCV(estimator, method=method, cv="prefit")
    cal.fit(X_cal, y_cal)
    return cal


def summarize(results: dict[str, dict[str, float]]) -> pd.DataFrame:
    """Ordena los resultados de varios modelos por PR-AUC."""
    df = pd.DataFrame(results).T
    cols = [c for c in ["pr_auc", "roc_auc", "lift@5", "lift@10", "capture@10",
                        "capture@20", "brier", "base_rate"] if c in df.columns]
    return df[cols].sort_values("pr_auc", ascending=False).round(4)

## 8 · Modelos

En datos tabulares de banca un GBM bien tuneado casi siempre le gana a un MLP. La red
neuronal aporta cuando se le da lo que el GBM no puede ver: la **secuencia mensual
cruda**. Por eso hay dos cosas distintas.

| | Modelos |
|:---|:---|
| Tabulares (`build_model_zoo`) | `logreg`, `random_forest`, `lightgbm`, `xgboost`, `catboost`, `mlp` |
| Secuencial (`SequenceModel`) | GRU de PyTorch sobre `[muestras, meses, canales]` |
| Combinación | Stacking de los top-k por CV, y blend GBM × GRU |

Cuáles corren lo decide `models.enabled`: `mlp` está definido pero fuera de esa
lista, así que no se entrena salvo que lo añadas.

Dos detalles que cuestan caro si no se saben de antemano: todos los estimadores van
con `n_jobs=1`, porque anidar paralelismo dentro y fuera del `RandomizedSearchCV`
produce un deadlock de joblib; y el tope de `C` de la logística es deliberadamente
bajo, porque con clases muy desbalanceadas y datos casi separables los coeficientes
se disparan y el `matmul` desborda a infinito.

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)


# ── Preprocesamiento ────────────────────────────────────────────────────

def _clip_extremes(X):
    """Recorta a ±10 sigmas tras el escalado.

    Una columna casi constante tiene desviación ~1e-160; al escalarla,
    StandardScaler la convierte en valores del orden de 1e160 y el `matmul`
    de la regresión logística desborda a inf. VarianceThreshold quita las
    constantes exactas y este recorte cubre el resto.
    """
    return np.clip(X, -10.0, 10.0)


def make_preprocessor(numeric: list[str], categorical: list[str],
                      for_trees: bool = True) -> ColumnTransformer:
    """Los árboles toleran NaN y prefieren codificación ordinal; los modelos
    lineales y las redes necesitan imputación, escala y one-hot."""
    if for_trees:
        num_tf = SimpleImputer(strategy="median", add_indicator=True)
        cat_tf = Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="__missing__")),
            ("enc", OrdinalEncoder(handle_unknown="use_encoded_value",
                                   unknown_value=-1)),
        ])
    else:
        num_tf = Pipeline([
            ("imp", SimpleImputer(strategy="median", add_indicator=True)),
            ("var", VarianceThreshold(threshold=1e-10)),
            ("sc", StandardScaler()),
            ("clip", FunctionTransformer(_clip_extremes, feature_names_out="one-to-one")),
        ])
        cat_tf = Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="__missing__")),
            ("enc", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01,
                                  sparse_output=False)),
        ])
    return ColumnTransformer(
        [("num", num_tf, numeric), ("cat", cat_tf, categorical)],
        remainder="drop", verbose_feature_names_out=False,
    )


# ── Zoo ─────────────────────────────────────────────────────────────────

def build_model_zoo(numeric: list[str], categorical: list[str],
                    scale_pos_weight: float = 1.0,
                    random_state: int = 42) -> dict[str, dict[str, Any]]:
    """Devuelve {nombre: {"pipeline": Pipeline, "params": search_space}}.

    Nota sobre paralelismo: todos los estimadores usan **un solo hilo**
    (`n_jobs=1`). El paralelismo va en el `RandomizedSearchCV` de afuera.
    Anidar `n_jobs=-1` dentro y fuera produce un deadlock de joblib/loky
    (el proceso se queda vivo al 0% de CPU) — un clásico difícil de
    diagnosticar si no se sabe de antemano.
    """
    zoo: dict[str, dict[str, Any]] = {}
    pre_lin = make_preprocessor(numeric, categorical, for_trees=False)
    pre_tree = make_preprocessor(numeric, categorical, for_trees=True)

    zoo["logreg"] = {
        "pipeline": Pipeline([
            ("pre", pre_lin),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                       random_state=random_state)),
        ]),
        "params": {
            # Tope de C deliberadamente bajo. Con clases muy desbalanceadas y
            # datos casi separables, una C grande deja que los coeficientes se
            # disparen (separación cuasi-completa) y el `matmul` desborda a inf.
            # La regularización tiene que morder siempre.
            "clf__C": loguniform(1e-4, 1e1),
            "clf__penalty": ["l2"],
            "clf__solver": ["lbfgs"],
        },
    }

    zoo["random_forest"] = {
        "pipeline": Pipeline([
            ("pre", pre_tree),
            ("clf", RandomForestClassifier(class_weight="balanced_subsample",
                                           n_jobs=1, random_state=random_state)),
        ]),
        "params": {
            "clf__n_estimators": randint(200, 700),
            "clf__max_depth": randint(4, 18),
            "clf__min_samples_leaf": randint(5, 80),
            "clf__max_features": uniform(0.2, 0.6),
        },
    }

    try:
        from lightgbm import LGBMClassifier

        zoo["lightgbm"] = {
            "pipeline": Pipeline([
                ("pre", pre_tree),
                ("clf", LGBMClassifier(objective="binary", n_jobs=1, verbose=-1,
                                       scale_pos_weight=scale_pos_weight,
                                       random_state=random_state)),
            ]),
            "params": {
                "clf__n_estimators": randint(200, 900),
                "clf__learning_rate": loguniform(0.01, 0.2),
                "clf__num_leaves": randint(15, 120),
                "clf__min_child_samples": randint(10, 150),
                "clf__subsample": uniform(0.6, 0.4),
                "clf__subsample_freq": [1],
                "clf__colsample_bytree": uniform(0.5, 0.5),
                "clf__reg_lambda": loguniform(1e-3, 20),
            },
        }
    except ImportError:
        pass

    try:
        from xgboost import XGBClassifier

        zoo["xgboost"] = {
            "pipeline": Pipeline([
                ("pre", pre_tree),
                ("clf", XGBClassifier(objective="binary:logistic", n_jobs=1,
                                      eval_metric="aucpr", tree_method="hist",
                                      scale_pos_weight=scale_pos_weight,
                                      random_state=random_state)),
            ]),
            "params": {
                "clf__n_estimators": randint(200, 900),
                "clf__learning_rate": loguniform(0.01, 0.2),
                "clf__max_depth": randint(3, 10),
                "clf__min_child_weight": randint(1, 20),
                "clf__subsample": uniform(0.6, 0.4),
                "clf__colsample_bytree": uniform(0.5, 0.5),
                "clf__reg_lambda": loguniform(1e-3, 20),
            },
        }
    except ImportError:
        pass

    try:
        from catboost import CatBoostClassifier

        zoo["catboost"] = {
            "pipeline": Pipeline([
                ("pre", pre_tree),
                ("clf", CatBoostClassifier(verbose=0, allow_writing_files=False,
                                           thread_count=1,
                                           loss_function="Logloss",
                                           scale_pos_weight=scale_pos_weight,
                                           random_seed=random_state)),
            ]),
            "params": {
                "clf__iterations": randint(200, 800),
                "clf__learning_rate": loguniform(0.01, 0.2),
                "clf__depth": randint(4, 9),
                "clf__l2_leaf_reg": loguniform(1, 30),
            },
        }
    except ImportError:
        pass

    from sklearn.neural_network import MLPClassifier

    zoo["mlp"] = {
        "pipeline": Pipeline([
            ("pre", pre_lin),
            ("clf", MLPClassifier(max_iter=400, early_stopping=True,
                                  random_state=random_state)),
        ]),
        "params": {
            "clf__hidden_layer_sizes": [(64,), (128,), (128, 64), (64, 32)],
            "clf__alpha": loguniform(1e-5, 1e-1),
            "clf__learning_rate_init": loguniform(1e-4, 1e-2),
        },
    }
    return zoo


# ── Tuning ──────────────────────────────────────────────────────────────

def tune(name: str, spec: dict[str, Any], X: pd.DataFrame, y: np.ndarray,
         groups: np.ndarray, cfg: Config, verbose: bool = True):
    """RandomizedSearchCV con StratifiedGroupKFold agrupado por CIF."""
    m = cfg.models
    search = RandomizedSearchCV(
        spec["pipeline"], spec["params"],
        n_iter=int(m.get("n_iter", 20)),
        scoring=m.get("scoring", "average_precision"),
        cv=make_cv(cfg),
        n_jobs=int(m.get("n_jobs", -1)),
        random_state=int(cfg.split.get("random_state", 42)),
        refit=True, error_score="raise",
    )
    search.fit(X, y, groups=groups)
    if verbose:
        print(f"   {name:14s} CV {m.get('scoring')} = {search.best_score_:.4f}")
    return search


def tune_all(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
             numeric: list[str], categorical: list[str], cfg: Config,
             verbose: bool = True) -> dict[str, Any]:
    """Tunea todos los modelos habilitados. Devuelve {nombre: search}."""
    pos = float(y.mean())
    spw = (1 - pos) / pos if 0 < pos < 1 else 1.0
    zoo = build_model_zoo(numeric, categorical, scale_pos_weight=spw,
                          random_state=int(cfg.split.get("random_state", 42)))
    enabled = [n for n in cfg.models.get("enabled", list(zoo)) if n in zoo]

    if verbose:
        print(f"\n🔍 Tuning {len(enabled)} modelos "
              f"({cfg.models.get('n_iter')} iteraciones, "
              f"{cfg.split.get('n_folds')} folds agrupados por CIF)")
        print(f"   scale_pos_weight = {spw:.1f}  (tasa base {pos:.2%})")

    out = {}
    for name in enabled:
        try:
            out[name] = tune(name, zoo[name], X, y, groups, cfg, verbose)
        except Exception as e:  # un modelo roto no debe tumbar el pipeline
            print(f"   ⚠️  {name}: falló ({type(e).__name__}: {e})")
    return out


def build_stack(searches: dict[str, Any], top_k: int, cfg: Config):
    """Stacking con meta-modelo logístico sobre los top-k por CV.

    `cv` interno agrupado no es posible en StackingClassifier sin groups, así
    que se usa un CV estratificado simple sobre las predicciones out-of-fold.
    El riesgo de leakage aquí es menor porque los base learners ya fueron
    seleccionados con CV agrupado.
    """
    ranked = sorted(searches.items(), key=lambda kv: kv[1].best_score_, reverse=True)
    chosen = ranked[:top_k]
    if len(chosen) < 2:
        return None, [n for n, _ in chosen]

    estimators = [(n, s.best_estimator_) for n, s in chosen]
    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced"),
        stack_method="predict_proba",
        cv=3, n_jobs=1, passthrough=False,
    )
    return stack, [n for n, _ in chosen]


# ── Red secuencial (GRU sobre las series mensuales) ─────────────────────

SEQUENCE_COLS = [
    "owned_balance", "txn_count_total", "loan_balance", "n_deposit_accounts",
    "login_count_mobile_month", "login_count_web_month",
    "has_direct_deposit", "fees_charged",
]


def build_sequences(state: pd.DataFrame, index_df: pd.DataFrame,
                    seq_cols: list[str], window: int,
                    stats: dict[str, tuple[float, float]] | None = None,
                    return_stats: bool = False):
    """Arma el tensor [n_muestras, ventana, n_canales].

    Para cada fila de `index_df` (cif, as_of_month) toma los `window` meses
    que terminan en ese mes — nunca posteriores.

    `stats` son las medias y desviaciones por canal con las que normalizar.
    Se calculan una sola vez sobre el train (`return_stats=True`) y se
    reutilizan en validación y test: si cada tensor se normalizara con las
    suyas, un saldo de $5.000 en test y uno de $1.000 en train acabarían en
    el mismo valor escalado, y el GRU vería un holdout desplazado.
    """
    cols = [c for c in seq_cols if c in state.columns]
    s = state.sort_values(["cif_id", "as_of_month"]).copy()

    lagged = {}
    g = s.groupby("cif_id")
    for c in cols:
        for lag in range(window):
            lagged[f"{c}__lag{lag}"] = g[c].shift(lag)
    lag_df = pd.DataFrame(lagged, index=s.index)
    lag_df[["cif_id", "as_of_month"]] = s[["cif_id", "as_of_month"]]

    merged = index_df[["cif_id", "as_of_month"]].merge(
        lag_df, on=["cif_id", "as_of_month"], how="left"
    )
    n = len(merged)
    arr = np.zeros((n, window, len(cols)), dtype=np.float32)
    for ci, c in enumerate(cols):
        block = merged[[f"{c}__lag{lag}" for lag in range(window)]].to_numpy(dtype=np.float32)
        # lag0 es el mes actual → invertir para que el eje temporal avance
        arr[:, :, ci] = block[:, ::-1]

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    # Normalización por canal (log1p en montos para domar las colas)
    used: dict[str, tuple[float, float]] = {}
    for ci, c in enumerate(cols):
        ch = arr[:, :, ci]
        if "balance" in c or "fees" in c or "amt" in c:
            ch = np.sign(ch) * np.log1p(np.abs(ch))
        if stats is not None and c in stats:
            mu, sd = stats[c]
        else:
            mu, sd = float(ch.mean()), float(ch.std())
            sd = sd if sd > 1e-6 else 1.0
        used[c] = (mu, sd)
        arr[:, :, ci] = (ch - mu) / sd
    return (arr, used) if return_stats else arr


class SequenceModel:
    """GRU binario sobre las series mensuales. API estilo sklearn (fit/predict_proba)."""

    def __init__(self, n_features: int, hidden_size: int = 64, num_layers: int = 1,
                 dropout: float = 0.2, lr: float = 1e-3, epochs: int = 30,
                 batch_size: int = 256, patience: int = 5, seed: int = 42):
        import torch
        import torch.nn as nn

        torch.manual_seed(seed)
        self.torch, self.nn = torch, nn
        self.epochs, self.batch_size, self.patience, self.lr = epochs, batch_size, patience, lr
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        class _Net(nn.Module):
            def __init__(self):
                super().__init__()
                self.gru = nn.GRU(n_features, hidden_size, num_layers=num_layers,
                                  batch_first=True,
                                  dropout=dropout if num_layers > 1 else 0.0)
                self.head = nn.Sequential(
                    nn.LayerNorm(hidden_size), nn.Dropout(dropout),
                    nn.Linear(hidden_size, 32), nn.ReLU(),
                    nn.Linear(32, 1),
                )

            def forward(self, x):
                out, _ = self.gru(x)
                return self.head(out[:, -1, :]).squeeze(-1)

        self.model = _Net().to(self.device)

    def fit(self, X: np.ndarray, y: np.ndarray, X_val=None, y_val=None, verbose=True):
        torch, nn = self.torch, self.nn
        from torch.utils.data import DataLoader, TensorDataset

        pos = float(y.mean())
        pos_weight = torch.tensor([(1 - pos) / pos if 0 < pos < 1 else 1.0],
                                  device=self.device)
        crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y.astype(np.float32)))
        dl = DataLoader(ds, batch_size=self.batch_size, shuffle=True)

        best, bad, best_state = np.inf, 0, None
        for ep in range(self.epochs):
            self.model.train()
            tot = 0.0
            for xb, yb in dl:
                xb, yb = xb.to(self.device), yb.to(self.device)
                opt.zero_grad()
                loss = crit(self.model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                opt.step()
                tot += float(loss) * len(xb)
            train_loss = tot / len(ds)

            if X_val is not None:
                from sklearn.metrics import average_precision_score
                p = self.predict_proba(X_val)[:, 1]
                score = -average_precision_score(y_val, p)
            else:
                score = train_loss

            if score < best - 1e-5:
                best, bad = score, 0
                best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            else:
                bad += 1
                if bad >= self.patience:
                    if verbose:
                        print(f"   early stopping en época {ep + 1}")
                    break
        if best_state is not None:
            self.model.load_state_dict(best_state)
        if verbose:
            metric = "PR-AUC val" if X_val is not None else "loss train"
            print(f"   GRU entrenado — mejor {metric}: {abs(best):.4f}")
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        torch = self.torch
        self.model.eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(X), 4096):
                xb = torch.from_numpy(X[i:i + 4096]).to(self.device)
                outs.append(torch.sigmoid(self.model(xb)).cpu().numpy())
        p = np.concatenate(outs) if outs else np.zeros(len(X))
        return np.column_stack([1 - p, p])

## 9 · Explicabilidad

`TreeExplainer` es exacto y determinista sobre GBMs.

**Por qué no LIME:** reajusta un modelo lineal local con muestreo aleatorio, así que
dos corridas sobre el mismo cliente devuelven explicaciones distintas. Inaceptable si
la explicación puede terminar en un expediente de model risk (SR 11-7).

Y SHAP dice *qué pesó*, no *qué hacer*: para eso está `counterfactual_analysis`, que
simula la intervención sobre los clientes de mayor riesgo y mide la caída de
probabilidad.

In [ ]:
# ── Utilidades de pipeline ──────────────────────────────────────────────

def _split_pipeline(estimator):
    """Separa (preprocesador, modelo final) de un Pipeline de sklearn."""
    if hasattr(estimator, "named_steps"):
        steps = list(estimator.named_steps.items())
        pre = estimator[:-1] if len(steps) > 1 else None
        model = steps[-1][1]
        return pre, model
    return None, estimator


def _transformed_frame(estimator, X: pd.DataFrame) -> pd.DataFrame:
    """Aplica el preprocesador y devuelve un DataFrame con nombres de columna."""
    pre, _ = _split_pipeline(estimator)
    if pre is None:
        return X
    Xt = pre.transform(X)
    try:
        names = list(pre.get_feature_names_out())
    except Exception:
        names = [f"f{i}" for i in range(Xt.shape[1])]
    if hasattr(Xt, "toarray"):
        Xt = Xt.toarray()
    return pd.DataFrame(np.asarray(Xt), columns=names, index=X.index)


# ── SHAP ────────────────────────────────────────────────────────────────

def shap_analysis(estimator, X: pd.DataFrame, cfg: Config | None = None,
                  sample_size: int | None = None, verbose: bool = True):
    """Calcula valores SHAP. Devuelve (shap_values, X_transformado).

    Usa TreeExplainer cuando el modelo final es de árboles (exacto y rápido);
    si no, cae a un explainer genérico con un background pequeño.
    """
    import shap

    n = sample_size or (cfg.explain.get("shap_sample_size", 2000) if cfg else 2000)
    Xs = X.sample(min(n, len(X)), random_state=42) if len(X) > n else X
    Xt = _transformed_frame(estimator, Xs)
    _, model = _split_pipeline(estimator)

    tree_like = any(k in type(model).__name__.lower()
                    for k in ("forest", "lgbm", "xgb", "catboost", "boosting", "tree"))
    if verbose:
        print(f"🔍 SHAP sobre {len(Xt):,} filas "
              f"({'TreeExplainer' if tree_like else 'Explainer genérico'})")

    if tree_like:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(Xt)
    else:
        bg = shap.sample(Xt, min(100, len(Xt)), random_state=42)
        explainer = shap.Explainer(model.predict_proba, bg)
        sv = explainer(Xt).values

    sv = np.asarray(sv)
    # Normalizar la forma: algunos backends devuelven (n, f, 2) o lista por clase
    if sv.ndim == 3:
        sv = sv[:, :, 1] if sv.shape[2] == 2 else sv[:, :, 0]
    return sv, Xt


def global_importance(shap_values: np.ndarray, Xt: pd.DataFrame,
                      top_n: int = 25) -> pd.DataFrame:
    """Importancia global: |SHAP| medio por feature, con la dirección del efecto."""
    mean_abs = np.abs(shap_values).mean(axis=0)
    # Correlación entre el valor de la feature y su contribución = dirección
    direction = []
    for i, c in enumerate(Xt.columns):
        col = Xt[c].to_numpy(dtype=float)
        if np.std(col) < 1e-9:
            direction.append(0.0)
        else:
            direction.append(float(np.corrcoef(col, shap_values[:, i])[0, 1]))

    out = pd.DataFrame({
        "feature": Xt.columns,
        "shap_abs_mean": mean_abs,
        "direccion": np.round(direction, 3),
    }).sort_values("shap_abs_mean", ascending=False).head(top_n)
    out["efecto"] = np.where(out["direccion"] > 0.05, "↑ sube riesgo",
                     np.where(out["direccion"] < -0.05, "↓ baja riesgo", "no lineal"))
    out["shap_abs_mean"] = out["shap_abs_mean"].round(5)
    return out.reset_index(drop=True)


def explain_customer(estimator, X: pd.DataFrame, row_index, top_n: int = 10,
                     shap_values: np.ndarray | None = None,
                     Xt: pd.DataFrame | None = None) -> pd.DataFrame:
    """Explicación local: los N drivers del score de un cliente concreto."""
    if shap_values is None or Xt is None:
        shap_values, Xt = shap_analysis(estimator, X.loc[[row_index]], verbose=False)
        pos = 0
    else:
        if row_index not in Xt.index:
            raise KeyError(f"{row_index} no está en la muestra SHAP; "
                           "recalcula con esa fila incluida.")
        pos = Xt.index.get_loc(row_index)

    contrib = shap_values[pos]
    out = pd.DataFrame({
        "feature": Xt.columns,
        "valor": Xt.iloc[pos].to_numpy(),
        "contribucion_shap": contrib,
    })
    out["abs"] = out["contribucion_shap"].abs()
    out = out.sort_values("abs", ascending=False).head(top_n).drop(columns="abs")
    out["direccion"] = np.where(out["contribucion_shap"] > 0, "↑ riesgo", "↓ riesgo")
    return out.reset_index(drop=True).round(4)


# ── Contrafactuales ─────────────────────────────────────────────────────

def counterfactual_analysis(estimator, X: pd.DataFrame, cfg: Config,
                            features: list[str] | None = None,
                            top_pct: float = 0.10,
                            verbose: bool = True) -> pd.DataFrame:
    """¿Cuánto baja el riesgo si se mueve una palanca accionable?

    Para cada feature accionable, simula la intervención sobre los clientes de
    mayor riesgo y mide la caída promedio en probabilidad. Es la traducción de
    "el modelo dice X" a "retención puede hacer Y".
    """
    feats = features or cfg.explain.get("counterfactual_features", [])
    feats = [f for f in feats if f in X.columns]
    if not feats:
        return pd.DataFrame(columns=["feature", "intervencion", "delta_riesgo_medio"])

    base_p = estimator.predict_proba(X)[:, 1]
    k = max(1, int(len(X) * top_pct))
    at_risk = np.argsort(base_p)[::-1][:k]
    Xr = X.iloc[at_risk]
    base_r = base_p[at_risk]

    rows = []
    for f in feats:
        col = X[f]
        if pd.api.types.is_bool_dtype(col) or set(col.dropna().unique()) <= {0, 1}:
            target, label = 1.0, f"activar {f}"
        else:
            target = float(col.quantile(0.75))
            label = f"subir {f} al p75 ({target:.1f})"

        Xc = Xr.copy()
        Xc[f] = target
        new_p = estimator.predict_proba(Xc)[:, 1]
        delta = float((new_p - base_r).mean())
        rows.append({
            "feature": f,
            "intervencion": label,
            "riesgo_antes": round(float(base_r.mean()), 4),
            "riesgo_despues": round(float(new_p.mean()), 4),
            "delta_riesgo_medio": round(delta, 4),
            "reduccion_%": round(-100 * delta / max(base_r.mean(), 1e-9), 1),
        })

    out = pd.DataFrame(rows).sort_values("delta_riesgo_medio")
    if verbose:
        print(f"\n🎯 Contrafactuales sobre el top {top_pct:.0%} de riesgo "
              f"({k:,} clientes)")
    return out.reset_index(drop=True)


# Identificadores y geografía. Los patrones van anclados a separadores de
# palabra a propósito: buscar "account" suelto marcaría `n_joint_accounts`, y
# "state" suelto marcaría `estatement_flag`. Una alerta que salta en cada
# corrida sobre features legítimas enseña a ignorarla, que es justo lo que no
# se quiere de un control de model risk.
_SUSPICIOUS = (
    r"(^|_)branch", r"(^|_)zip", r"(^|_)state($|_)",
    r"(^|_)cif", r"(^|_)household", r"_id($|_)",
)


def sanity_check(importance: pd.DataFrame, expected_top: list[str] | None = None,
                 verbose: bool = True) -> dict[str, bool]:
    """Valida el SHAP contra intuición de negocio.

    Si aparece un identificador o una variable geográfica arriba, casi siempre
    es un proxy de algo — o leakage. Vale más detectarlo aquí que en el comité
    de validación.
    """
    top = importance["feature"].head(10).str.lower().tolist()
    suspicious = [f for f in top if any(re.search(p, f) for p in _SUSPICIOUS)]
    checks = {
        "sin_proxies_geograficos_o_ids_en_top10": len(suspicious) == 0,
    }
    if expected_top:
        found = [e for e in expected_top if any(e.lower() in t for t in top)]
        checks["recupera_señales_esperadas"] = len(found) > 0
        if verbose and found:
            print(f"✓ Señales esperadas en el top-10: {found}")
    if verbose and suspicious:
        print(f"⚠️  Features sospechosas en el top-10 (posible proxy o leakage): "
              f"{suspicious}")
    return checks

# Parte II — El análisis

A partir de aquí se ejecuta. Cada sección deja a la vista lo que hay que mirar antes
de dar por bueno el paso siguiente.

## 10 · Carga y validación

`load_tables` devuelve las 7 tablas con el mismo esquema, vengan de donde vengan.
`validate` las contrasta contra el contrato declarado en `churn/config.py`:

- llaves duplicadas,
- columnas obligatorias faltantes,
- nulos por encima del umbral tolerado.

In [ ]:
tables = load_tables(cfg)
profile(tables)

In [ ]:
_ = validate(tables)

## 11 · EDA

Tres cosas que hay que mirar **antes** de modelar. Ninguna es decorativa: cada una
fija un parámetro concreto del pipeline.

| Gráfico | Qué decide |
|:---|:---|
| Aperturas y cierres por mes | Si hubo una conversión de core, y dónde cortar la historia |
| Distribución de saldos | El umbral de saldo bajo del target (`target.low_balance_abs`) |
| Mix de productos y roles | Cuánto pesan las cuentas joint y los authorized signers |

> **Un pico aislado de cierres casi nunca es comportamiento de clientes.** Suele ser
> una migración de core. Si aparece, corta la historia ahí en lugar de dejar que el
> modelo aprenda el artefacto.

In [ ]:
# ── 1. Aperturas vs cierres por mes: detector de conversión de core ──
ev = tables["events"]
opens = (ev[ev.event_type == "account_open"].groupby(ev.event_date.dt.to_period("M"))
         .size().rename("aperturas"))
closes = (ev[ev.event_type == "account_close"].groupby(ev.event_date.dt.to_period("M"))
          .size().rename("cierres"))
flow = pd.concat([opens, closes], axis=1).fillna(0)
flow = flow[flow.index >= tables["deposits"].as_of_month.min().to_period("M")]

fig, ax = plt.subplots(figsize=(12, 4))
flow.plot(ax=ax, marker="o", ms=3)
ax.set_title("Aperturas y cierres por mes — un pico aislado = posible conversión de core")
ax.set_xlabel(""); ax.legend()
plt.tight_layout(); plt.show()

z = (flow.cierres - flow.cierres.mean()) / flow.cierres.std()
if (z.abs() > 3).any():
    print(f"⚠️  Meses con cierres anómalos (|z|>3): {list(flow.index[z.abs() > 3])}")
else:
    print("✓ Sin picos anómalos de cierres")

In [ ]:
# ── 2. Distribución de saldos: de aquí sale el umbral del target ──
dep = tables["deposits"]
bal = dep.loc[dep.eom_balance > 0, "eom_balance"]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(np.log10(bal.clip(lower=1)), bins=60, color="steelblue")
axes[0].set_title("log10(saldo) — cuentas con saldo > 0"); axes[0].set_xlabel("log10 USD")
for p in (0.5, 1, 2, 3):
    axes[0].axvline(p, color="crimson", ls="--", lw=0.8)

qs = bal.quantile([.05, .10, .25, .50, .75, .90, .99])
axes[1].axis("off")
axes[1].table(cellText=[[f"p{int(i*100)}", f"${v:,.0f}"] for i, v in qs.items()],
              colLabels=["percentil", "saldo"], loc="center", cellLoc="left")
axes[1].set_title("Percentiles de saldo")
plt.tight_layout(); plt.show()

print(f"Cuentas con saldo < $100:  {(dep.eom_balance < 100).mean():.1%}")
print(f"Cuentas con saldo == 0:    {(dep.eom_balance == 0).mean():.1%}")
print(f"\n→ El umbral `target.low_balance_abs` (hoy "
      f"${cfg.target['low_balance_abs']:,.0f}) debe caer en la cola izquierda,")
print("  no cortar clientes normales.")

In [ ]:
# ── 3. Mix de productos y de roles del CIF ──
rel = tables["relationships"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

dep.product_family.value_counts().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Cuentas-mes por familia de producto")

rc = rel.relationship_role.value_counts()
rc.plot.barh(ax=axes[1], color="darkorange")
axes[1].set_title("Relaciones por rol del CIF")
plt.tight_layout(); plt.show()

share = (rc / rc.sum()).mul(100).round(1)
print("Peso de cada rol (%):")
print(share.to_string())
print(f"\n→ {share.get('joint', 0):.1f}% joint: si sumaras saldos sin ponderar por")
print("  ownership, inflarías los depósitos del banco en esa proporción.")

## 12 · Panel y target

Aquí se materializan las tres decisiones de diseño del modelo.

**Saldos ponderados por rol.** Una cuenta joint no cuenta entera para cada titular
(`1/n`), y un authorized signer no cuenta nada (peso `0`). Sin esto, los depósitos
del banco quedan inflados por doble conteo.

**Target compuesto.** No basta con el cierre formal: la attrition silenciosa
—saldo bajo **y** sin préstamos **y** sin actividad— sostenida durante
`target.persistence_months` cuenta igual. Es la que se escapa de los reportes.

**La espina es el maestro de clientes**, no las relaciones vigentes. Si se
construyera sobre relaciones, el cliente que cierra todo desaparecería del panel
justo en el mes que interesa, en vez de aparecer con ceros.

In [ ]:
state = build_cif_month_state(tables, cfg)
state_lab = label_attrition_state(state, cfg)

print(f"Estado CIF×mes: {state.shape[0]:,} filas × {state.shape[1]} columnas")
n = state_lab.cif_id.nunique()
ev_n = state_lab.dropna(subset=["churn_event_month"]).cif_id.nunique()
print(f"CIFs con evento de churn detectado: {ev_n:,} / {n:,} ({ev_n/n:.1%})")

In [ ]:
# Descomposición del target: ¿qué condición está mandando?
# El umbral de inactividad se deriva de CONFIG igual que en
# label_attrition_state, para que esta tabla no se desincronice del target.
inact_m = max(1, round(cfg.target["inactivity_days"] / 30))

conds = {
    "saldo bajo": state_lab.owned_balance < np.maximum(
        cfg.target["low_balance_abs"],
        state_lab.balance_baseline_12m.fillna(0) * cfg.target["low_balance_pct_of_baseline"]),
    "sin préstamos activos": state_lab.n_active_loans.fillna(0) <= 0,
    f"inactivo ≥{inact_m}m": state_lab.months_since_activity.fillna(0) >= inact_m,
    "sin relaciones vigentes": state_lab.n_relationships_any_role.fillna(0) <= 0,
    "ESTADO attrited (combinado)": state_lab.is_attrited_state,
}
pd.Series({k: f"{v.mean():.2%}" for k, v in conds.items()}, name="% de filas del panel").to_frame()

In [ ]:
# Se reusa el estado ya etiquetado de la celda anterior: reconstruirlo aquí
# duplicaría la parte más lenta del pipeline.
panel = build_panel(tables, cfg, state=state_lab)

# Estabilidad de la tasa base en el tiempo: si salta, hay un problema de datos
rate = panel.groupby("as_of_month").y.agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(rate.index, rate["mean"] * 100, marker="o", ms=4, color="crimson")
ax.axhline(panel.y.mean() * 100, ls="--", color="gray", lw=1,
           label=f"promedio {panel.y.mean():.2%}")
ax.set_title("Tasa de churn por mes de observación"); ax.set_ylabel("% churn")
ax.legend(); plt.tight_layout(); plt.show()

## 13 · Feature engineering

Para cada fila `(cif_id, as_of_month)`, **todo** se calcula sobre la ventana hacia
atrás `[t-11, t]`. Nunca sobre el futuro.

Nueve familias de señal:

| Familia | Qué captura |
|:---|:---|
| Nivel | El valor de hoy — saldo, productos, logins |
| Tendencia | Pendiente OLS sobre la ventana |
| Ratio a sí mismo | El cliente contra su propia línea base, no contra la media del banco |
| Drawdown | Caída desde el máximo de la ventana |
| Volatilidad | Dispersión de la serie |
| Aceleración | Cambio de la pendiente: si se desacelera o se desploma |
| Rachas | Meses consecutivos en una misma condición |
| Estacionalidad | Patrones recurrentes por mes del año |
| Relacionales | Contagio — el co-titular que ya se está yendo |

In [ ]:
import re

feats = build_features(panel, cfg, tables)
numeric, categorical = select_feature_columns(feats)
feats = clean_features(feats, numeric)

print(f"{len(numeric)} numéricas + {len(categorical)} categóricas")
print("\nMuestra por familia:")
for fam, pat in [("nivel",       r"^(owned_balance|txn_count_total|loan_balance|n_deposit_accounts)$"),
                 ("tendencia",   r"_slope_\d+m$"),
                 ("drawdown",    r"_drawdown_\d+m$"),
                 ("aceleración", r"_accel$"),
                 ("racha",       r"_declining_streak$|^months_since_"),
                 ("dominio",     r"dd_lost|loan_ending|cd_maturing"),
                 ("relacional",  r"co_owner|n_joint_accounts|n_signer_only")]:
    cols = [c for c in numeric if re.search(pat, c)]
    print(f"  {fam:14s} {len(cols):3d}  ej: {cols[:3]}")

## 14 · Split out-of-time con embargo

Dos protecciones contra leakage, en este orden:

1. **Corte temporal, no aleatorio.** Los últimos `split.oot_months` meses son el
   holdout. Un split aleatorio pondría el futuro dentro del train.
2. **Embargo.** Se eliminan del train las filas cuya ventana de etiqueta invade el
   periodo de test.

El embargo cuesta `gap + performance` meses de panel. Es caro y aun así no es
opcional: sin él, el holdout está contaminado y las métricas mienten hacia arriba.

In [ ]:
train, test = temporal_split(feats, cfg)

Xtr, ytr = train[numeric + categorical], train.y.to_numpy()
Xte, yte = test[numeric + categorical], test.y.to_numpy()
groups = train.cif_id.to_numpy()   # ← clave: el CV agrupa por esto

## 15 · Tuning con `StratifiedGroupKFold`

`groups=cif_id` es lo que impide que el mismo cliente esté a la vez en train y en
validación. Con un `StratifiedKFold` normal el PR-AUC sube varios puntos sin que el
modelo haya mejorado en nada.

La métrica de optimización es **average precision (PR-AUC)** — ni accuracy ni
ROC-AUC. Con una tasa base de 2-5%, predecir "no se va nadie" acierta el 96% de
las veces y no sirve para nada.

In [ ]:
%%time
searches = tune_all(Xtr, ytr, groups, numeric, categorical, cfg)

# `preds` guarda la probabilidad de cada candidato sobre el test. Es lo que
# permite más adelante puntuar exactamente el modelo que encabeza la tabla,
# incluidos los que no son un estimador de sklearn (GRU, blend).
results, fitted, preds = {}, {}, {}
for name, s in searches.items():
    preds[name] = s.best_estimator_.predict_proba(Xte)[:, 1]
    results[name] = evaluate(yte, preds[name])
    fitted[name] = s.best_estimator_

summarize(results)

## 16 · Stacking

Los `models.stack_top_k` mejores modelos por PR-AUC de CV entran como base learners,
con una regresión logística como meta-modelo sobre sus probabilidades out-of-fold.

In [ ]:
stack, chosen = build_stack(searches, cfg.models["stack_top_k"], cfg)
if stack is not None:
    print(f"Base learners: {chosen}")
    stack.fit(Xtr, ytr)
    preds["stacking"] = stack.predict_proba(Xte)[:, 1]
    results["stacking"] = evaluate(yte, preds["stacking"])
    fitted["stacking"] = stack
summarize(results)

## 17 · Red secuencial (GRU)

El GBM ve *resúmenes* de la trayectoria: pendientes, drawdowns, rachas. El GRU ve la
**serie mensual cruda** —`[muestras, meses, canales]`— y aprende la forma completa de
la caída.

Ahí está el valor real del ensemble: mezclar dos maneras distintas de mirar la misma
historia, no promediar cinco GBMs que se equivocan en los mismos clientes.

> **El early stopping y el peso de la mezcla se deciden en una partición de
> validación agrupada por CIF, sacada del train.** Usar el test para eso —elegir la
> época que mejor puntúa en el holdout y luego reportar ese holdout— infla las
> métricas por la misma puerta que el embargo cierra.

In [ ]:
%%time
from sklearn.model_selection import GroupShuffleSplit

sq = cfg.models["sequence_model"]

if not sq.get("enabled", True):
    print("ℹ️  Red secuencial desactivada en la configuración "
          "(models.sequence_model.enabled = false) — se omite.")
else:
    win = cfg.panel["feature_window_months"]

    # Validación DENTRO del train, agrupada por CIF igual que el CV de los
    # modelos tabulares. Aquí se deciden el early stopping del GRU y el peso
    # del blend: hacerlo contra el test sería elegir hiperparámetros mirando
    # el holdout, que es justo lo que el embargo trata de impedir.
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2,
                            random_state=cfg.split["random_state"])
    i_tr, i_val = next(gss.split(train, train.y, groups=train.cif_id))
    sub_tr, sub_val = train.iloc[i_tr], train.iloc[i_val]
    y_sub, y_val = sub_tr.y.to_numpy(), sub_val.y.to_numpy()

    # Las estadísticas de normalización salen solo del train y se reutilizan
    # en validación y test, para que un mismo saldo signifique lo mismo en los
    # tres tensores.
    Str, seq_stats = build_sequences(state, sub_tr, SEQUENCE_COLS, win, return_stats=True)
    Sval = build_sequences(state, sub_val, SEQUENCE_COLS, win, stats=seq_stats)
    Ste = build_sequences(state, test, SEQUENCE_COLS, win, stats=seq_stats)
    print(f"Tensor: {Str.shape}  [muestras, meses, canales]   "
          f"validación: {Sval.shape[0]:,} filas")

    gru = SequenceModel(n_features=Str.shape[2], hidden_size=sq["hidden_size"],
                        num_layers=sq["num_layers"], dropout=sq["dropout"],
                        lr=sq["lr"], epochs=sq["epochs"], batch_size=sq["batch_size"],
                        patience=sq["patience"]).fit(Str, y_sub, Sval, y_val)

    preds["gru_sequence"] = gru.predict_proba(Ste)[:, 1]
    results["gru_sequence"] = evaluate(yte, preds["gru_sequence"])

    # El tabular base se elige por su score de CV, no por el test; el peso de
    # la mezcla, en la partición de validación.
    best_tab = max(searches, key=lambda k: searches[k].best_score_)
    p_tab_val = fitted[best_tab].predict_proba(sub_val[numeric + categorical])[:, 1]
    p_gru_val = gru.predict_proba(Sval)[:, 1]
    w = max((0.0, 0.1, 0.2, 0.3, 0.4, 0.5),
            key=lambda w: evaluate(y_val, (1 - w) * p_tab_val + w * p_gru_val)["pr_auc"])
    print(f"Base tabular: {best_tab}   peso del GRU elegido en validación: {w:.0%}")

    if w > 0:
        blend = f"blend_{best_tab}+gru_{w:.0%}"
        preds[blend] = (1 - w) * preds[best_tab] + w * preds["gru_sequence"]
        results[blend] = evaluate(yte, preds[blend])

summarize(results)

## 18 · Resultados

**Cómo leer la tabla: ignora `accuracy`.** Con una tasa base de 2-5% no dice nada.

| Métrica | Qué significa |
|:---|:---|
| `pr_auc` | Métrica principal — área bajo la curva precision-recall |
| `lift@10` | Cuántas veces mejor que aleatorio en el top 10% de riesgo |
| `capture@10` | Qué fracción de **todos** los churners cae en ese top 10% |
| `brier` | Qué tan bien calibradas están las probabilidades |

Retención solo puede llamar a una lista corta, así que `lift@10` y `capture@10` son
las que gobiernan la decisión de negocio.

In [ ]:
board = summarize(results)
best_name = board.index[0]
p_best = preds[best_name]          # exactamente el modelo que encabeza la tabla


# SHAP y contrafactuales necesitan un estimador de sklearn, y TreeExplainer
# solo es exacto sobre árboles. El GRU y el blend no lo son, y el stacking
# caería en el explainer genérico (muestreo, lento y no determinista). Así que
# la explicabilidad corre sobre el mejor GBM de la tabla, dicho explícitamente.
def _is_tree(est):
    final = est[-1] if hasattr(est, "named_steps") else est
    return any(k in type(final).__name__.lower()
               for k in ("forest", "lgbm", "xgb", "catboost", "boosting"))


explain_name = next((n for n in board.index if n in fitted and _is_tree(fitted[n])),
                    next(n for n in board.index if n in fitted))
explain_model = fitted[explain_name]
p_explain = preds[explain_name]

print(f"🏆 {best_name}: PR-AUC {board.loc[best_name,'pr_auc']:.4f}, "
      f"lift@10 {board.loc[best_name,'lift@10']:.1f}x")
if explain_name != best_name:
    print(f"   Explicabilidad y calibración sobre {explain_name} "
          f"(PR-AUC {board.loc[explain_name,'pr_auc']:.4f})")
board

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

dt = decile_table(yte, p_best)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].bar(dt.decil, dt.lift, color="steelblue")
axes[0].axhline(1, color="crimson", ls="--", lw=1)
axes[0].set_title("Lift por decil"); axes[0].set_xlabel("decil (1 = mayor riesgo)")

axes[1].plot(dt.decil * 10, dt["capture_%"], marker="o", color="darkorange")
axes[1].plot([0, 100], [0, 100], ls="--", color="gray", lw=1, label="aleatorio")
axes[1].set_title("Curva de captura"); axes[1].set_xlabel("% de la base contactada")
axes[1].set_ylabel("% de churners capturados"); axes[1].legend()

pr, rc, _ = precision_recall_curve(yte, p_best)
axes[2].plot(rc, pr, color="seagreen")
axes[2].axhline(yte.mean(), ls="--", color="gray", lw=1, label=f"base {yte.mean():.2%}")
axes[2].set_title("Curva Precision-Recall"); axes[2].set_xlabel("recall")
axes[2].set_ylabel("precision"); axes[2].legend()

plt.tight_layout(); plt.show()
dt

## 19 · Calibración

Necesaria en cuanto la probabilidad alimente un cálculo de valor en riesgo
(`P(churn) × saldo × margen`): ahí ya no basta con ordenar bien, el número tiene que
significar lo que dice. El stacking suele descalibrar.

In [ ]:
# El calibrador se ajusta en el primer tercio del test y se mide en el resto,
# así nunca se evalúa sobre filas que ya vio. Como el panel viene ordenado
# por CIF, el corte cae entre clientes: solo el que queda justo en el límite
# aparece en ambos tramos.
n_cal = len(Xte) // 3
cal = calibrate(explain_model, Xte.iloc[:n_cal], yte[:n_cal])

y_ev = yte[n_cal:]
p_raw = p_explain[n_cal:]                              # mismo modelo, sin calibrar
p_cal = cal.predict_proba(Xte.iloc[n_cal:])[:, 1]

before, after = calibration_table(y_ev, p_raw), calibration_table(y_ev, p_cal)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(before.prob_predicha, before.tasa_observada, "o-", label="sin calibrar")
ax.plot(after.prob_predicha, after.tasa_observada, "s-", label="isotónica")
lim = max(before.prob_predicha.max(), after.prob_predicha.max()) * 1.1
ax.plot([0, lim], [0, lim], "--", color="gray", lw=1, label="perfecta")
ax.set_xlabel("probabilidad predicha"); ax.set_ylabel("tasa observada")
ax.set_title(f"Curva de calibración — {explain_name}"); ax.legend()
plt.tight_layout(); plt.show()

print(f"Brier: {evaluate(y_ev, p_raw)['brier']:.5f} → "
      f"{evaluate(y_ev, p_cal)['brier']:.5f}")

## 20 · Explicabilidad (SHAP)

`TreeExplainer` es exacto y determinista sobre GBMs. Por eso esta sección corre
sobre el mejor modelo **de árboles** de la tabla, que no siempre es el que la
encabeza: sobre un stacking o un blend habría que caer al explainer genérico, que
muestrea y deja de ser reproducible. La celda anterior imprime cuál se usó.

**Por qué no LIME:** reajusta un modelo lineal local con muestreo aleatorio, así que
dos corridas sobre el mismo cliente devuelven explicaciones distintas. Inaceptable si
la explicación puede terminar en un expediente de model risk (SR 11-7).

`sanity_check` avisa si aparece una variable geográfica o un identificador en el
top-10: casi siempre es proxy de algo, o leakage.

In [ ]:
sv, Xt = shap_analysis(explain_model, Xte, cfg)
imp = global_importance(sv, Xt, top_n=25)

fig, ax = plt.subplots(figsize=(9, 7))
top = imp.head(20).iloc[::-1]
colors = ["crimson" if d > 0.05 else "steelblue" if d < -0.05 else "gray"
          for d in top.direccion]
ax.barh(top.feature, top.shap_abs_mean, color=colors)
ax.set_xlabel("|SHAP| medio"); ax.set_title(f"Importancia global — {explain_name} (rojo = sube riesgo)")
plt.tight_layout(); plt.show()

_ = sanity_check(imp, expected_top=["direct_deposit", "balance", "drawdown", "slope"])
imp

In [ ]:
# Explicación local: un cliente concreto del top de riesgo.
# Se ordena por p_explain (el modelo que produjo el SHAP), no por p_best, para
# que el cliente señalado y la explicación vengan del mismo modelo.
top_idx = Xte.index[np.argsort(p_explain)[::-1][:1]][0]

if top_idx in Xt.index:
    display(explain_customer(explain_model, Xte, top_idx, top_n=12,
                             shap_values=sv, Xt=Xt))
else:
    # No cayó en la muestra de `shap_sample_size`. TreeExplainer es exacto fila
    # a fila, así que se calcula solo para esta en vez de renunciar: si no, la
    # explicación local desaparece cada vez que el sorteo no lo incluye.
    display(explain_customer(explain_model, Xte, top_idx, top_n=12))

## 21 · Contrafactuales accionables

SHAP dice *qué pesó*. Retención necesita *qué hacer*. Esta celda simula la
intervención sobre los clientes de mayor riesgo —moviendo las variables de
`explain.counterfactual_features`— y mide cuánto baja la probabilidad.

In [ ]:
cf = counterfactual_analysis(explain_model, Xte, cfg)
cf

## 22 · Lista de scoring y guardado

Lo que se entrega al equipo de retención: el top de riesgo con su probabilidad
**calibrada** y el saldo en riesgo asociado. Sale del tramo de test que el calibrador
no llegó a ver, para que el número sea comparable entre clientes y sumable.

Se guardan además el leaderboard, la importancia SHAP global y los dos modelos —el
crudo y el calibrado—.

In [ ]:
import joblib

# La lista sale del tramo de test que el calibrador no vio, y con la
# probabilidad ya calibrada: sin eso, P(churn) × saldo no es comparable
# entre clientes ni sumable como valor en riesgo.
scored = test.iloc[n_cal:][["cif_id", "as_of_month", "owned_balance", "y"]].copy()
scored["p_churn"] = p_cal
scored["valor_en_riesgo"] = scored.p_churn * scored.owned_balance
scored = scored.sort_values("p_churn", ascending=False)

out = cfg.artifacts_path()
scored.head(500).to_csv(out / "lista_retencion_top500.csv", index=False)
board.to_csv(out / "leaderboard.csv")
imp.to_csv(out / "shap_importance.csv", index=False)
joblib.dump(explain_model, out / "modelo_final.joblib")
joblib.dump(cal, out / "modelo_calibrado.joblib")

print(f"💾 Guardado en {out}  (modelo: {explain_name})")
print(f"\nValor total en riesgo (top 10%): "
      f"${scored.head(len(scored)//10).valor_en_riesgo.sum():,.0f}")
scored.head(15)

---

## Siguiente paso

1. Cambia `cfg.data["source"] = "drive"` y sube los extractos del agente SQL.
2. Revisa la EDA de **saldos** y ajusta `target.low_balance_abs` en la celda de configuración a lo
   que digan tus datos — no lo inventes.
3. Revisa el gráfico de **aperturas/cierres**: si hay un pico, corta la historia ahí.
4. Corre la descomposición del target. Si una sola condición domina, la definición
   está desbalanceada.
5. Antes de presentar: valida el SHAP contra intuición de negocio y corre un análisis
   de proxies sobre `state` y `preferred_language` (Fair Lending / UDAAP).